# VSCode GPU Matbench Full Rerun

This notebook is the clean rerun entrypoint for the project when VSCode is connected to a CUDA GPU runtime. It uses official Matbench folds, a single Magpie feature pipeline, consistent model/preprocessing definitions, and one unified output directory per run.

Scientific controls used here:

- official Matbench train/test folds only,
- no test-fold tuning,
- same feature matrix per task for every model,
- dummy and linear baselines included,
- strong tree baselines included,
- TabPFN evaluated against the best non-TabPFN baseline,
- fold-level metrics saved before aggregation,
- all figures use the same style and color mapping.


## 1. Locate project and load config

Run this notebook from VSCode with the project folder open. The output for each rerun is written to `results/runs/<run_id>/`, and `results/latest` points to the most recent run.

In [ ]:
from pathlib import Path
import copy
import os
import sys

try:
    import yaml
except ModuleNotFoundError:
    yaml = None

DEFAULT_CONFIG = {
    'run': {
        'base_dir': 'results/runs',
        'clean_existing_run_dir': False,
        'update_latest_pointer': True,
        'primary_metric': 'mae',
        'random_seed': 42,
    },
    'tasks': [
        'matbench_steels',
        'matbench_jdft2d',
        'matbench_phonons',
        'matbench_expt_gap',
    ],
    'models': [
        'dummy_mean',
        'ridge_cv',
        'random_forest',
        'extra_trees',
        'hist_gradient_boosting',
        'tabpfn',
    ],
    'features': {
        'feature_sets': ['magpie', 'magpie_structure'],
        'use_cache': True,
        'show_progress': True,
        'n_jobs': 1,
    },
    'classical_models': {'n_estimators': 500},
    'tabpfn': {
        'device': 'cuda',
        'n_estimators': 8,
        'predict_batch_size': 128,
        'kwargs': {
            'inference_precision': 'auto',
            'memory_saving_mode': 'auto',
            'show_progress_bar': False,
        },
    },
}

# These files let the notebook run on a remote VSCode/Colab kernel even when
# the Mac project folder is not mounted on the GPU runtime filesystem.
BOOTSTRAP_FILES = {'configs/gpu_rerun.yml': 'run:\n'
                          '  base_dir: results/runs\n'
                          '  clean_existing_run_dir: false\n'
                          '  update_latest_pointer: true\n'
                          '  primary_metric: mae\n'
                          '  random_seed: 42\n'
                          '\n'
                          'tasks:\n'
                          '  - matbench_steels\n'
                          '  - matbench_jdft2d\n'
                          '  - matbench_phonons\n'
                          '  - matbench_expt_gap\n'
                          '\n'
                          'models:\n'
                          '  - dummy_mean\n'
                          '  - ridge_cv\n'
                          '  - random_forest\n'
                          '  - extra_trees\n'
                          '  - hist_gradient_boosting\n'
                          '  - tabpfn\n'
                          '\n'
                          'features:\n'
                          '  feature_sets:\n'
                          '    - magpie\n'
                          '    - magpie_structure\n'
                          '  use_cache: true\n'
                          '  show_progress: true\n'
                          '  n_jobs: 1\n'
                          '  note: magpie_structure is only run for structure-input '
                          'tasks and combines Magpie composition features with '
                          'density, symmetry, packing, and structural-complexity '
                          'descriptors.\n'
                          '\n'
                          'classical_models:\n'
                          '  n_estimators: 500\n'
                          '\n'
                          'tabpfn:\n'
                          '  device: cuda\n'
                          '  n_estimators: 8\n'
                          '  predict_batch_size: 128\n'
                          '  kwargs:\n'
                          '    inference_precision: auto\n'
                          '    memory_saving_mode: auto\n'
                          '    show_progress_bar: false\n',
 'results/figures/.gitkeep': '',
 'results/metrics/.gitkeep': '',
 'results/predictions/.gitkeep': '',
 'results/runs/.gitkeep': '',
 'src/matbench_tabpfn/__init__.py': '"""Reusable utilities for the Matbench TabPFN '
                                    'project."""\n'
                                    '\n'
                                    'from .paths import RunPaths, create_run_paths\n'
                                    '\n'
                                    '__all__ = ["RunPaths", "create_run_paths"]\n',
 'src/matbench_tabpfn/analysis.py': '"""Automatic result summaries and error-analysis '
                                    'tables."""\n'
                                    '\n'
                                    'from __future__ import annotations\n'
                                    '\n'
                                    'from pathlib import Path\n'
                                    '\n'
                                    'import pandas as pd\n'
                                    '\n'
                                    'from .paths import RunPaths\n'
                                    '\n'
                                    '\n'
                                    'def best_models_by_task(summary_df: pd.DataFrame) '
                                    '-> pd.DataFrame:\n'
                                    '    if summary_df.empty:\n'
                                    '        return pd.DataFrame()\n'
                                    '    return (\n'
                                    '        summary_df.sort_values(["task", '
                                    '"mean_mae"])\n'
                                    '        .groupby("task", as_index=False)\n'
                                    '        .first()\n'
                                    '        .sort_values("task")\n'
                                    '        .reset_index(drop=True)\n'
                                    '    )\n'
                                    '\n'
                                    '\n'
                                    'def structure_feature_branch_summary(summary_df: '
                                    'pd.DataFrame) -> pd.DataFrame:\n'
                                    '    if summary_df.empty or "feature_set" not in '
                                    'summary_df.columns:\n'
                                    '        return pd.DataFrame()\n'
                                    '\n'
                                    '    data = summary_df.query("matbench_input_type '
                                    '== \'structure\'").copy()\n'
                                    '    if data.empty or '
                                    'data["feature_set"].nunique() < 2:\n'
                                    '        return pd.DataFrame()\n'
                                    '\n'
                                    '    best = (\n'
                                    '        data.sort_values("mean_mae")\n'
                                    '        .groupby(["task", "feature_set", '
                                    '"feature_set_display"], as_index=False)\n'
                                    '        .first()\n'
                                    '    )\n'
                                    '    rows = []\n'
                                    '    for task_name, task_df in '
                                    'best.groupby("task", sort=False):\n'
                                    '        proxy = task_df.query("feature_set == '
                                    '\'magpie\'")\n'
                                    '        structure = task_df.query("feature_set == '
                                    '\'magpie_structure\'")\n'
                                    '        if proxy.empty or structure.empty:\n'
                                    '            continue\n'
                                    '        proxy_row = proxy.iloc[0]\n'
                                    '        structure_row = structure.iloc[0]\n'
                                    '        delta = structure_row["mean_mae"] - '
                                    'proxy_row["mean_mae"]\n'
                                    '        pct = 100.0 * delta / '
                                    'proxy_row["mean_mae"]\n'
                                    '        rows.append(\n'
                                    '            {\n'
                                    '                "task": task_name,\n'
                                    '                "proxy_best_model": '
                                    'proxy_row["model"],\n'
                                    '                "proxy_best_model_display": '
                                    'proxy_row["model_display"],\n'
                                    '                "proxy_mean_mae": '
                                    'proxy_row["mean_mae"],\n'
                                    '                "structure_best_model": '
                                    'structure_row["model"],\n'
                                    '                "structure_best_model_display": '
                                    'structure_row["model_display"],\n'
                                    '                "structure_mean_mae": '
                                    'structure_row["mean_mae"],\n'
                                    '                '
                                    '"mae_delta_structure_minus_proxy": delta,\n'
                                    '                '
                                    '"mae_pct_change_structure_minus_proxy": pct,\n'
                                    '            }\n'
                                    '        )\n'
                                    '    return pd.DataFrame(rows)\n'
                                    '\n'
                                    '\n'
                                    'def top_absolute_errors(\n'
                                    '    predictions_df: pd.DataFrame,\n'
                                    '    summary_df: pd.DataFrame,\n'
                                    '    *,\n'
                                    '    top_n: int = 10,\n'
                                    ') -> pd.DataFrame:\n'
                                    '    if predictions_df.empty or summary_df.empty:\n'
                                    '        return pd.DataFrame()\n'
                                    '\n'
                                    '    best = '
                                    'best_models_by_task(summary_df)[["task", '
                                    '"feature_set", "model"]]\n'
                                    '    rows = []\n'
                                    '    for _, best_row in best.iterrows():\n'
                                    '        task_name = best_row["task"]\n'
                                    '        feature_set = best_row["feature_set"]\n'
                                    '        model_name = best_row["model"]\n'
                                    '        subset = predictions_df.query(\n'
                                    '            "task == @task_name and feature_set '
                                    '== @feature_set and model == @model_name"\n'
                                    '        ).copy()\n'
                                    '        if subset.empty:\n'
                                    '            continue\n'
                                    '        subset = '
                                    'subset.sort_values("absolute_error", '
                                    'ascending=False).head(top_n)\n'
                                    '        subset["analysis_group"] = '
                                    '"best_model_by_task"\n'
                                    '        rows.append(subset)\n'
                                    '    return pd.concat(rows, ignore_index=True) if '
                                    'rows else pd.DataFrame()\n'
                                    '\n'
                                    '\n'
                                    'def tabpfn_vs_best_baseline_sample_errors(\n'
                                    '    predictions_df: pd.DataFrame,\n'
                                    '    baseline_comparison_df: pd.DataFrame,\n'
                                    '    *,\n'
                                    '    top_n: int = 20,\n'
                                    ') -> pd.DataFrame:\n'
                                    '    if predictions_df.empty or '
                                    'baseline_comparison_df.empty:\n'
                                    '        return pd.DataFrame()\n'
                                    '\n'
                                    '    rows = []\n'
                                    '    tabpfn_rows = '
                                    'baseline_comparison_df.query("model == '
                                    '\'tabpfn\'").copy()\n'
                                    '    for _, row in tabpfn_rows.iterrows():\n'
                                    '        tabpfn_pred = predictions_df.query(\n'
                                    '            "task == @row.task and feature_set == '
                                    '@row.feature_set and model == \'tabpfn\'"\n'
                                    '        ).copy()\n'
                                    '        baseline_pred = predictions_df.query(\n'
                                    '            "task == @row.task and "\n'
                                    '            "feature_set == '
                                    '@row.best_baseline_feature_set and "\n'
                                    '            "model == @row.best_baseline_model"\n'
                                    '        ).copy()\n'
                                    '        if tabpfn_pred.empty or '
                                    'baseline_pred.empty:\n'
                                    '            continue\n'
                                    '\n'
                                    '        merged = tabpfn_pred.merge(\n'
                                    '            baseline_pred[\n'
                                    '                [\n'
                                    '                    "task",\n'
                                    '                    "fold",\n'
                                    '                    "mbid",\n'
                                    '                    "feature_set",\n'
                                    '                    "model",\n'
                                    '                    "y_pred",\n'
                                    '                    "absolute_error",\n'
                                    '                ]\n'
                                    '            ],\n'
                                    '            on=["task", "fold", "mbid"],\n'
                                    '            how="inner",\n'
                                    '            suffixes=("_tabpfn", "_baseline"),\n'
                                    '        )\n'
                                    '        if merged.empty:\n'
                                    '            continue\n'
                                    '\n'
                                    '        merged = merged.rename(\n'
                                    '            columns={\n'
                                    '                "feature_set_tabpfn": '
                                    '"tabpfn_feature_set",\n'
                                    '                "feature_set_baseline": '
                                    '"baseline_feature_set",\n'
                                    '                "model_baseline": '
                                    '"baseline_model",\n'
                                    '                "y_pred_tabpfn": '
                                    '"tabpfn_y_pred",\n'
                                    '                "y_pred_baseline": '
                                    '"baseline_y_pred",\n'
                                    '                "absolute_error_tabpfn": '
                                    '"tabpfn_absolute_error",\n'
                                    '                "absolute_error_baseline": '
                                    '"baseline_absolute_error",\n'
                                    '            }\n'
                                    '        )\n'
                                    '        '
                                    'merged["tabpfn_error_minus_baseline_error"] = (\n'
                                    '            merged["tabpfn_absolute_error"] - '
                                    'merged["baseline_absolute_error"]\n'
                                    '        )\n'
                                    '        merged["tabpfn_error_ratio_vs_baseline"] '
                                    '= (\n'
                                    '            merged["tabpfn_absolute_error"] / '
                                    'merged["baseline_absolute_error"].replace(0, '
                                    'pd.NA)\n'
                                    '        )\n'
                                    '        rows.append(\n'
                                    '            merged.sort_values(\n'
                                    '                '
                                    '"tabpfn_error_minus_baseline_error", '
                                    'ascending=False\n'
                                    '            ).head(top_n)\n'
                                    '        )\n'
                                    '\n'
                                    '    return pd.concat(rows, ignore_index=True) if '
                                    'rows else pd.DataFrame()\n'
                                    '\n'
                                    '\n'
                                    'def write_auto_summary_markdown(\n'
                                    '    summary_df: pd.DataFrame,\n'
                                    '    baseline_comparison_df: pd.DataFrame,\n'
                                    '    structure_summary_df: pd.DataFrame,\n'
                                    '    path: Path,\n'
                                    ') -> str:\n'
                                    '    lines = ["# Automatic Results Summary", ""]\n'
                                    '\n'
                                    '    best = best_models_by_task(summary_df)\n'
                                    '    if not best.empty:\n'
                                    '        lines.extend(["## Best Model by Task", '
                                    '""])\n'
                                    '        for _, row in best.iterrows():\n'
                                    '            lines.append(\n'
                                    '                "- `{task}`: {model} with '
                                    '{feature} features, "\n'
                                    '                "MAE={mae:.4g} {unit}, '
                                    'R2={r2:.3g}.".format(\n'
                                    '                    task=row["task"],\n'
                                    '                    model=row["model_display"],\n'
                                    '                    '
                                    'feature=row["feature_set_display"],\n'
                                    '                    mae=row["mean_mae"],\n'
                                    '                    unit=row["unit"],\n'
                                    '                    r2=row["mean_r2"],\n'
                                    '                )\n'
                                    '            )\n'
                                    '        lines.append("")\n'
                                    '\n'
                                    '    if baseline_comparison_df.empty or "model" '
                                    'not in baseline_comparison_df.columns:\n'
                                    '        tabpfn = pd.DataFrame()\n'
                                    '    else:\n'
                                    '        tabpfn = '
                                    'baseline_comparison_df.query("model == '
                                    '\'tabpfn\'").copy()\n'
                                    '    if not tabpfn.empty:\n'
                                    '        lines.extend(["## TabPFN Claim Check", '
                                    '""])\n'
                                    '        for _, row in tabpfn.sort_values(["task", '
                                    '"feature_set"]).iterrows():\n'
                                    '            direction = "improves over" if '
                                    'row["mae_delta_vs_best_baseline"] < 0 else "does '
                                    'not beat"\n'
                                    '            lines.append(\n'
                                    '                "- `{task}` ({feature}): TabPFN '
                                    '{direction} the best non-TabPFN "\n'
                                    '                "baseline ({baseline}, '
                                    '{baseline_feature}) by {pct:+.2f}% MAE.".format(\n'
                                    '                    task=row["task"],\n'
                                    '                    '
                                    'feature=row["feature_set_display"],\n'
                                    '                    direction=direction,\n'
                                    '                    '
                                    'baseline=row["best_baseline_display"],\n'
                                    '                    '
                                    'baseline_feature=row["best_baseline_feature_set_display"],\n'
                                    '                    '
                                    'pct=row["mae_pct_change_vs_best_baseline"],\n'
                                    '                )\n'
                                    '            )\n'
                                    '        lines.append("")\n'
                                    '\n'
                                    '    if not structure_summary_df.empty:\n'
                                    '        lines.extend(["## Structure-Aware Feature '
                                    'Check", ""])\n'
                                    '        for _, row in '
                                    'structure_summary_df.sort_values("task").iterrows():\n'
                                    '            direction = "improves over" if '
                                    'row["mae_delta_structure_minus_proxy"] < 0 else '
                                    '"does not improve over"\n'
                                    '            lines.append(\n'
                                    '                "- `{task}`: best structure-aware '
                                    'branch {direction} the composition-proxy "\n'
                                    '                "branch by {pct:+.2f}% '
                                    'MAE.".format(\n'
                                    '                    task=row["task"],\n'
                                    '                    direction=direction,\n'
                                    '                    '
                                    'pct=row["mae_pct_change_structure_minus_proxy"],\n'
                                    '                )\n'
                                    '            )\n'
                                    '        lines.append("")\n'
                                    '\n'
                                    '    text = "\\n".join(lines).strip() + "\\n"\n'
                                    '    path.parent.mkdir(parents=True, '
                                    'exist_ok=True)\n'
                                    '    path.write_text(text, encoding="utf-8")\n'
                                    '    return text\n'
                                    '\n'
                                    '\n'
                                    'def create_report_artifacts(\n'
                                    '    *,\n'
                                    '    metrics_df: pd.DataFrame,\n'
                                    '    predictions_df: pd.DataFrame,\n'
                                    '    summary_df: pd.DataFrame,\n'
                                    '    baseline_comparison_df: pd.DataFrame,\n'
                                    '    paired_comparison_df: pd.DataFrame,\n'
                                    '    paths: RunPaths,\n'
                                    '    top_n_errors: int = 10,\n'
                                    ') -> dict[str, Path | str]:\n'
                                    '    """Save tables and markdown that convert raw '
                                    'metrics into report evidence."""\n'
                                    '\n'
                                    '    paths.tables.mkdir(parents=True, '
                                    'exist_ok=True)\n'
                                    '\n'
                                    '    best_df = best_models_by_task(summary_df)\n'
                                    '    structure_df = '
                                    'structure_feature_branch_summary(summary_df)\n'
                                    '    top_errors_df = top_absolute_errors(\n'
                                    '        predictions_df, summary_df, '
                                    'top_n=top_n_errors\n'
                                    '    )\n'
                                    '    tabpfn_errors_df = '
                                    'tabpfn_vs_best_baseline_sample_errors(\n'
                                    '        predictions_df, baseline_comparison_df, '
                                    'top_n=top_n_errors\n'
                                    '    )\n'
                                    '\n'
                                    '    outputs: dict[str, Path | str] = {}\n'
                                    '    tables = {\n'
                                    '        "best_models_by_task": best_df,\n'
                                    '        "structure_feature_branch_summary": '
                                    'structure_df,\n'
                                    '        "top_absolute_errors": top_errors_df,\n'
                                    '        "tabpfn_vs_best_baseline_sample_errors": '
                                    'tabpfn_errors_df,\n'
                                    '        "paired_fold_comparisons": '
                                    'paired_comparison_df,\n'
                                    '    }\n'
                                    '\n'
                                    '    for name, table in tables.items():\n'
                                    '        path = paths.tables / f"{name}.csv"\n'
                                    '        table.to_csv(path, index=False)\n'
                                    '        outputs[name] = path\n'
                                    '\n'
                                    '    summary_text = write_auto_summary_markdown(\n'
                                    '        summary_df,\n'
                                    '        baseline_comparison_df,\n'
                                    '        structure_df,\n'
                                    '        paths.tables / "auto_summary.md",\n'
                                    '    )\n'
                                    '    outputs["auto_summary"] = paths.tables / '
                                    '"auto_summary.md"\n'
                                    '    outputs["auto_summary_text"] = summary_text\n'
                                    '    return outputs\n',
 'src/matbench_tabpfn/evaluation.py': '"""Official-fold evaluation utilities."""\n'
                                      '\n'
                                      'from __future__ import annotations\n'
                                      '\n'
                                      'import time\n'
                                      'from pathlib import Path\n'
                                      'from typing import Any, Sequence\n'
                                      '\n'
                                      'import numpy as np\n'
                                      'import pandas as pd\n'
                                      'from sklearn.metrics import '
                                      'mean_absolute_error, r2_score\n'
                                      '\n'
                                      'from .features import build_task_features, '
                                      'load_matbench_task\n'
                                      'from .models import build_model, '
                                      'predict_in_batches\n'
                                      'from .paths import RunPaths, write_json\n'
                                      'from .settings import MODEL_DISPLAY_NAMES, '
                                      'RANDOM_SEED\n'
                                      '\n'
                                      '\n'
                                      'def summarize_metrics(metrics_df: pd.DataFrame) '
                                      '-> pd.DataFrame:\n'
                                      '    ok = metrics_df.query("status == '
                                      '\'ok\'").copy()\n'
                                      '    if ok.empty:\n'
                                      '        return pd.DataFrame()\n'
                                      '\n'
                                      '    summary = (\n'
                                      '        ok.groupby(\n'
                                      '            [\n'
                                      '                "task",\n'
                                      '                "target",\n'
                                      '                "unit",\n'
                                      '                "matbench_input_type",\n'
                                      '                "feature_source",\n'
                                      '                "feature_set",\n'
                                      '                "feature_set_display",\n'
                                      '                "model",\n'
                                      '                "model_display",\n'
                                      '            ],\n'
                                      '            dropna=False,\n'
                                      '        )\n'
                                      '        .agg(\n'
                                      '            folds_completed=("fold", '
                                      '"nunique"),\n'
                                      '            mean_mae=("mae", "mean"),\n'
                                      '            std_mae=("mae", "std"),\n'
                                      '            sem_mae=("mae", lambda x: '
                                      'x.std(ddof=1) / np.sqrt(len(x))),\n'
                                      '            mean_r2=("r2", "mean"),\n'
                                      '            std_r2=("r2", "std"),\n'
                                      '            n_features=("n_features", "mean"),\n'
                                      '            train_size_mean=("train_size", '
                                      '"mean"),\n'
                                      '            test_size_sum=("test_size", '
                                      '"sum"),\n'
                                      '            fit_seconds_sum=("fit_seconds", '
                                      '"sum"),\n'
                                      '            '
                                      'predict_seconds_sum=("predict_seconds", '
                                      '"sum"),\n'
                                      '        )\n'
                                      '        .reset_index()\n'
                                      '    )\n'
                                      '    summary["rank_by_mae"] = '
                                      'summary.groupby("task")["mean_mae"].rank(\n'
                                      '        method="dense", ascending=True\n'
                                      '    ).astype(int)\n'
                                      '    return summary.sort_values(["task", '
                                      '"rank_by_mae", '
                                      '"model"]).reset_index(drop=True)\n'
                                      '\n'
                                      '\n'
                                      'def compare_against_best_baseline(summary_df: '
                                      'pd.DataFrame) -> pd.DataFrame:\n'
                                      '    if summary_df.empty:\n'
                                      '        return pd.DataFrame()\n'
                                      '\n'
                                      '    rows: list[dict[str, Any]] = []\n'
                                      '    for task_name, task_summary in '
                                      'summary_df.groupby("task", sort=False):\n'
                                      '        baseline_pool = task_summary[\n'
                                      '            '
                                      '~task_summary["model"].isin(["tabpfn", '
                                      '"dummy_mean"])\n'
                                      '        ].copy()\n'
                                      '        if baseline_pool.empty:\n'
                                      '            continue\n'
                                      '        best = '
                                      'baseline_pool.sort_values("mean_mae").iloc[0]\n'
                                      '        for _, row in task_summary.iterrows():\n'
                                      '            delta = row["mean_mae"] - '
                                      'best["mean_mae"]\n'
                                      '            pct = 100.0 * delta / '
                                      'best["mean_mae"]\n'
                                      '            rows.append(\n'
                                      '                {\n'
                                      '                    "task": task_name,\n'
                                      '                    "feature_source": '
                                      'row["feature_source"],\n'
                                      '                    "feature_set": '
                                      'row["feature_set"],\n'
                                      '                    "feature_set_display": '
                                      'row["feature_set_display"],\n'
                                      '                    "model": row["model"],\n'
                                      '                    "model_display": '
                                      'row["model_display"],\n'
                                      '                    "mean_mae": '
                                      'row["mean_mae"],\n'
                                      '                    "best_baseline_model": '
                                      'best["model"],\n'
                                      '                    "best_baseline_display": '
                                      'best["model_display"],\n'
                                      '                    '
                                      '"best_baseline_feature_set": '
                                      'best["feature_set"],\n'
                                      '                    '
                                      '"best_baseline_feature_set_display": '
                                      'best["feature_set_display"],\n'
                                      '                    "best_baseline_mean_mae": '
                                      'best["mean_mae"],\n'
                                      '                    '
                                      '"mae_delta_vs_best_baseline": delta,\n'
                                      '                    '
                                      '"mae_pct_change_vs_best_baseline": pct,\n'
                                      '                }\n'
                                      '            )\n'
                                      '    return pd.DataFrame(rows)\n'
                                      '\n'
                                      '\n'
                                      'def paired_fold_comparisons(metrics_df: '
                                      'pd.DataFrame) -> pd.DataFrame:\n'
                                      '    ok = metrics_df.query("status == '
                                      '\'ok\'").copy()\n'
                                      '    if ok.empty or "tabpfn" not in '
                                      'set(ok["model"]):\n'
                                      '        return pd.DataFrame()\n'
                                      '\n'
                                      '    rows: list[dict[str, Any]] = []\n'
                                      '    tabpfn_rows = ok.query("model == '
                                      '\'tabpfn\'").copy()\n'
                                      '    for (task_name, reference_feature_set), '
                                      'tabpfn_metrics in tabpfn_rows.groupby(\n'
                                      '        ["task", "feature_set"], sort=False\n'
                                      '    ):\n'
                                      '        task_metrics = ok.query("task == '
                                      '@task_name").copy()\n'
                                      '        tabpfn = tabpfn_metrics[["fold", '
                                      '"mae"]].rename(columns={"mae": "tabpfn_mae"})\n'
                                      '        baseline_pool = task_metrics[\n'
                                      '            '
                                      '~task_metrics["model"].isin(["tabpfn", '
                                      '"dummy_mean"])\n'
                                      '        ].copy()\n'
                                      '        if tabpfn.empty or '
                                      'baseline_pool.empty:\n'
                                      '            continue\n'
                                      '\n'
                                      '        baseline_summary = (\n'
                                      '            baseline_pool.groupby(["model", '
                                      '"feature_set"])["mae"]\n'
                                      '            .mean()\n'
                                      '            .sort_values()\n'
                                      '            .reset_index()\n'
                                      '        )\n'
                                      '        best_baseline_model = '
                                      'baseline_summary.iloc[0]["model"]\n'
                                      '        best_baseline_feature_set = '
                                      'baseline_summary.iloc[0]["feature_set"]\n'
                                      '        baseline = baseline_pool.query(\n'
                                      '            "model == @best_baseline_model and '
                                      'feature_set == @best_baseline_feature_set"\n'
                                      '        )[["fold", '
                                      '"mae"]].rename(columns={"mae": '
                                      '"baseline_mae"})\n'
                                      '        merged = tabpfn.merge(baseline, '
                                      'on="fold", how="inner")\n'
                                      '        if merged.empty:\n'
                                      '            continue\n'
                                      '\n'
                                      '        merged["delta_tabpfn_minus_baseline"] = '
                                      '(\n'
                                      '            merged["tabpfn_mae"] - '
                                      'merged["baseline_mae"]\n'
                                      '        )\n'
                                      '        rows.append(\n'
                                      '            {\n'
                                      '                "task": task_name,\n'
                                      '                "reference_model": "tabpfn",\n'
                                      '                "reference_feature_set": '
                                      'reference_feature_set,\n'
                                      '                "best_baseline_model": '
                                      'best_baseline_model,\n'
                                      '                "best_baseline_feature_set": '
                                      'best_baseline_feature_set,\n'
                                      '                "n_paired_folds": len(merged),\n'
                                      '                '
                                      '"mean_delta_tabpfn_minus_baseline": merged[\n'
                                      '                    '
                                      '"delta_tabpfn_minus_baseline"\n'
                                      '                ].mean(),\n'
                                      '                '
                                      '"median_delta_tabpfn_minus_baseline": merged[\n'
                                      '                    '
                                      '"delta_tabpfn_minus_baseline"\n'
                                      '                ].median(),\n'
                                      '                "tabpfn_win_folds": '
                                      'int((merged["tabpfn_mae"] < '
                                      'merged["baseline_mae"]).sum()),\n'
                                      '            }\n'
                                      '        )\n'
                                      '    return pd.DataFrame(rows)\n'
                                      '\n'
                                      '\n'
                                      'def _save_combined_outputs(\n'
                                      '    paths: RunPaths,\n'
                                      '    metrics: list[dict[str, Any]],\n'
                                      '    predictions: list[pd.DataFrame],\n'
                                      ') -> tuple[pd.DataFrame, pd.DataFrame, '
                                      'pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n'
                                      '    metrics_df = pd.DataFrame(metrics)\n'
                                      '    predictions_df = (\n'
                                      '        pd.concat(predictions, '
                                      'ignore_index=True) if predictions else '
                                      'pd.DataFrame()\n'
                                      '    )\n'
                                      '    summary_df = summarize_metrics(metrics_df)\n'
                                      '    baseline_comparison_df = '
                                      'compare_against_best_baseline(summary_df)\n'
                                      '    paired_comparison_df = '
                                      'paired_fold_comparisons(metrics_df)\n'
                                      '\n'
                                      '    metrics_df.to_csv(paths.metrics / '
                                      '"fold_metrics.csv", index=False)\n'
                                      '    predictions_df.to_csv(paths.predictions / '
                                      '"all_predictions.csv", index=False)\n'
                                      '    summary_df.to_csv(paths.metrics / '
                                      '"model_summary.csv", index=False)\n'
                                      '    baseline_comparison_df.to_csv(\n'
                                      '        paths.metrics / '
                                      '"best_baseline_comparison.csv", index=False\n'
                                      '    )\n'
                                      '    paired_comparison_df.to_csv(paths.metrics / '
                                      '"paired_fold_comparisons.csv", index=False)\n'
                                      '\n'
                                      '    return (\n'
                                      '        metrics_df,\n'
                                      '        predictions_df,\n'
                                      '        summary_df,\n'
                                      '        baseline_comparison_df,\n'
                                      '        paired_comparison_df,\n'
                                      '    )\n'
                                      '\n'
                                      '\n'
                                      'def run_official_fold_experiment(\n'
                                      '    *,\n'
                                      '    tasks: Sequence[str],\n'
                                      '    models: Sequence[str],\n'
                                      '    paths: RunPaths,\n'
                                      '    feature_sets: Sequence[str] | None = None,\n'
                                      '    random_seed: int = RANDOM_SEED,\n'
                                      '    n_estimators: int = 500,\n'
                                      '    tabpfn_n_estimators: int = 8,\n'
                                      '    tabpfn_device: str = "cuda",\n'
                                      '    tabpfn_predict_batch_size: int | None = '
                                      '128,\n'
                                      '    use_feature_cache: bool = True,\n'
                                      '    show_feature_progress: bool = True,\n'
                                      '    feature_n_jobs: int = 1,\n'
                                      '    continue_on_error: bool = False,\n'
                                      '    tabpfn_kwargs: dict[str, Any] | None = '
                                      'None,\n'
                                      ') -> tuple[pd.DataFrame, pd.DataFrame, '
                                      'pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n'
                                      '    """Run selected models on official Matbench '
                                      'folds and save tidy outputs."""\n'
                                      '\n'
                                      '    metrics: list[dict[str, Any]] = []\n'
                                      '    predictions: list[pd.DataFrame] = []\n'
                                      '    feature_sets = list(feature_sets or '
                                      '["magpie"])\n'
                                      '\n'
                                      '    for task_name in tasks:\n'
                                      '        task_start = time.time()\n'
                                      '        task = load_matbench_task(task_name)\n'
                                      '        target_col = task.metadata["target"]\n'
                                      '        unit = task.metadata.get("unit")\n'
                                      '        input_type = '
                                      'task.metadata["input_type"]\n'
                                      '\n'
                                      '        for feature_set in feature_sets:\n'
                                      '            if feature_set == '
                                      '"magpie_structure" and input_type != '
                                      '"structure":\n'
                                      '                print(f"{task_name} | '
                                      '{feature_set} | skipped; task is not structure '
                                      'input")\n'
                                      '                continue\n'
                                      '\n'
                                      '            feature_start = time.time()\n'
                                      '            features, feature_source, '
                                      'feature_set_display = build_task_features(\n'
                                      '                task_name,\n'
                                      '                task,\n'
                                      '                feature_set,\n'
                                      '                paths.features,\n'
                                      '                use_cache=use_feature_cache,\n'
                                      '                '
                                      'show_progress=show_feature_progress,\n'
                                      '                n_jobs=feature_n_jobs,\n'
                                      '            )\n'
                                      '\n'
                                      '            write_json(\n'
                                      '                paths.logs / '
                                      'f"{task_name}_{feature_set}_metadata.json",\n'
                                      '                {\n'
                                      '                    "task": task_name,\n'
                                      '                    "target": target_col,\n'
                                      '                    "unit": unit,\n'
                                      '                    "matbench_input_type": '
                                      'input_type,\n'
                                      '                    "feature_source": '
                                      'feature_source,\n'
                                      '                    "feature_set": '
                                      'feature_set,\n'
                                      '                    "feature_set_display": '
                                      'feature_set_display,\n'
                                      '                    "n_samples": len(task.df),\n'
                                      '                    "n_features": '
                                      'features.shape[1],\n'
                                      '                    "folds": '
                                      'list(task.folds_nums),\n'
                                      '                },\n'
                                      '            )\n'
                                      '\n'
                                      '            for model_name in models:\n'
                                      '                model_predictions: '
                                      'list[pd.DataFrame] = []\n'
                                      '                model_metrics: list[dict[str, '
                                      'Any]] = []\n'
                                      '\n'
                                      '                for fold in task.folds_nums:\n'
                                      '                    print(f"{task_name} | '
                                      '{feature_set} | {model_name} | fold {fold}")\n'
                                      '                    X_train_raw, y_train = '
                                      'task.get_train_and_val_data(fold)\n'
                                      '                    X_test_raw, y_test = '
                                      'task.get_test_data(fold, include_target=True)\n'
                                      '                    X_train = '
                                      'features.loc[X_train_raw.index]\n'
                                      '                    X_test = '
                                      'features.loc[X_test_raw.index]\n'
                                      '\n'
                                      '                    try:\n'
                                      '                        model = build_model(\n'
                                      '                            model_name,\n'
                                      '                            '
                                      'random_seed=random_seed,\n'
                                      '                            '
                                      'n_estimators=n_estimators,\n'
                                      '                            '
                                      'tabpfn_n_estimators=tabpfn_n_estimators,\n'
                                      '                            '
                                      'tabpfn_device=tabpfn_device,\n'
                                      '                            '
                                      'tabpfn_kwargs=tabpfn_kwargs,\n'
                                      '                        )\n'
                                      '                        fit_start = '
                                      'time.time()\n'
                                      '                        model.fit(X_train, '
                                      'y_train)\n'
                                      '                        fit_seconds = '
                                      'time.time() - fit_start\n'
                                      '\n'
                                      '                        predict_start = '
                                      'time.time()\n'
                                      '                        batch_size = (\n'
                                      '                            '
                                      'tabpfn_predict_batch_size if model_name == '
                                      '"tabpfn" else None\n'
                                      '                        )\n'
                                      '                        y_pred = '
                                      'predict_in_batches(model, X_test, batch_size)\n'
                                      '                        predict_seconds = '
                                      'time.time() - predict_start\n'
                                      '\n'
                                      '                        fold_mae = '
                                      'mean_absolute_error(y_test, y_pred)\n'
                                      '                        fold_r2 = '
                                      'r2_score(y_test, y_pred)\n'
                                      '\n'
                                      '                        row = {\n'
                                      '                            "status": "ok",\n'
                                      '                            "task": task_name,\n'
                                      '                            "target": '
                                      'target_col,\n'
                                      '                            "unit": unit,\n'
                                      '                            '
                                      '"matbench_input_type": input_type,\n'
                                      '                            "feature_source": '
                                      'feature_source,\n'
                                      '                            "feature_set": '
                                      'feature_set,\n'
                                      '                            '
                                      '"feature_set_display": feature_set_display,\n'
                                      '                            "model": '
                                      'model_name,\n'
                                      '                            "model_display": '
                                      'MODEL_DISPLAY_NAMES.get(\n'
                                      '                                model_name, '
                                      'model_name\n'
                                      '                            ),\n'
                                      '                            "fold": fold,\n'
                                      '                            "train_size": '
                                      'len(X_train),\n'
                                      '                            "test_size": '
                                      'len(X_test),\n'
                                      '                            "n_features": '
                                      'features.shape[1],\n'
                                      '                            "mae": fold_mae,\n'
                                      '                            "r2": fold_r2,\n'
                                      '                            "fit_seconds": '
                                      'fit_seconds,\n'
                                      '                            "predict_seconds": '
                                      'predict_seconds,\n'
                                      '                            "random_seed": '
                                      'random_seed,\n'
                                      '                        }\n'
                                      '                        '
                                      'model_metrics.append(row)\n'
                                      '                        metrics.append(row)\n'
                                      '\n'
                                      '                        fold_predictions = '
                                      'pd.DataFrame(\n'
                                      '                            {\n'
                                      '                                "task": '
                                      'task_name,\n'
                                      '                                "target": '
                                      'target_col,\n'
                                      '                                "unit": unit,\n'
                                      '                                '
                                      '"feature_source": feature_source,\n'
                                      '                                "feature_set": '
                                      'feature_set,\n'
                                      '                                '
                                      '"feature_set_display": feature_set_display,\n'
                                      '                                "model": '
                                      'model_name,\n'
                                      '                                '
                                      '"model_display": MODEL_DISPLAY_NAMES.get(\n'
                                      '                                    model_name, '
                                      'model_name\n'
                                      '                                ),\n'
                                      '                                "fold": fold,\n'
                                      '                                "mbid": '
                                      'y_test.index,\n'
                                      '                                "y_true": '
                                      'y_test.to_numpy(),\n'
                                      '                                "y_pred": '
                                      'y_pred,\n'
                                      '                                '
                                      '"absolute_error": np.abs(y_test.to_numpy() - '
                                      'y_pred),\n'
                                      '                            }\n'
                                      '                        )\n'
                                      '                        '
                                      'model_predictions.append(fold_predictions)\n'
                                      '                        '
                                      'predictions.append(fold_predictions)\n'
                                      '\n'
                                      '                    except Exception as exc:\n'
                                      '                        error_row = {\n'
                                      '                            "status": "error",\n'
                                      '                            "task": task_name,\n'
                                      '                            "target": '
                                      'target_col,\n'
                                      '                            "unit": unit,\n'
                                      '                            '
                                      '"matbench_input_type": input_type,\n'
                                      '                            "feature_source": '
                                      'feature_source,\n'
                                      '                            "feature_set": '
                                      'feature_set,\n'
                                      '                            '
                                      '"feature_set_display": feature_set_display,\n'
                                      '                            "model": '
                                      'model_name,\n'
                                      '                            "model_display": '
                                      'MODEL_DISPLAY_NAMES.get(\n'
                                      '                                model_name, '
                                      'model_name\n'
                                      '                            ),\n'
                                      '                            "fold": fold,\n'
                                      '                            "train_size": '
                                      'len(X_train),\n'
                                      '                            "test_size": '
                                      'len(X_test),\n'
                                      '                            "n_features": '
                                      'features.shape[1],\n'
                                      '                            "mae": np.nan,\n'
                                      '                            "r2": np.nan,\n'
                                      '                            "fit_seconds": '
                                      'np.nan,\n'
                                      '                            "predict_seconds": '
                                      'np.nan,\n'
                                      '                            "random_seed": '
                                      'random_seed,\n'
                                      '                            "error_type": '
                                      'type(exc).__name__,\n'
                                      '                            "error_message": '
                                      'str(exc),\n'
                                      '                        }\n'
                                      '                        '
                                      'model_metrics.append(error_row)\n'
                                      '                        '
                                      'metrics.append(error_row)\n'
                                      '                        '
                                      '_save_combined_outputs(paths, metrics, '
                                      'predictions)\n'
                                      '                        if not '
                                      'continue_on_error:\n'
                                      '                            raise\n'
                                      '\n'
                                      '                if model_metrics:\n'
                                      '                    '
                                      'pd.DataFrame(model_metrics).to_csv(\n'
                                      '                        paths.metrics\n'
                                      '                        / '
                                      'f"{task_name}_{feature_set}_{model_name}_fold_metrics.csv",\n'
                                      '                        index=False,\n'
                                      '                    )\n'
                                      '                if model_predictions:\n'
                                      '                    '
                                      'pd.concat(model_predictions, '
                                      'ignore_index=True).to_csv(\n'
                                      '                        paths.predictions\n'
                                      '                        / '
                                      'f"{task_name}_{feature_set}_{model_name}_predictions.csv",\n'
                                      '                        index=False,\n'
                                      '                    )\n'
                                      '                _save_combined_outputs(paths, '
                                      'metrics, predictions)\n'
                                      '\n'
                                      '            write_json(\n'
                                      '                paths.logs / '
                                      'f"{task_name}_{feature_set}_run_timing.json",\n'
                                      '                {\n'
                                      '                    "task": task_name,\n'
                                      '                    "feature_set": '
                                      'feature_set,\n'
                                      '                    "elapsed_seconds": '
                                      'time.time() - feature_start,\n'
                                      '                },\n'
                                      '            )\n'
                                      '\n'
                                      '        write_json(\n'
                                      '            paths.logs / '
                                      'f"{task_name}_run_timing.json",\n'
                                      '            {"task": task_name, '
                                      '"elapsed_seconds": time.time() - task_start},\n'
                                      '        )\n'
                                      '\n'
                                      '    return _save_combined_outputs(paths, '
                                      'metrics, predictions)\n',
 'src/matbench_tabpfn/features.py': '"""Matbench loading and feature generation."""\n'
                                    '\n'
                                    'from __future__ import annotations\n'
                                    '\n'
                                    'from pathlib import Path\n'
                                    '\n'
                                    'import numpy as np\n'
                                    'import pandas as pd\n'
                                    'from matbench.bench import MatbenchBenchmark\n'
                                    'from matminer.featurizers.composition import '
                                    'ElementProperty\n'
                                    'from matminer.featurizers.structure import (\n'
                                    '    DensityFeatures,\n'
                                    '    GlobalSymmetryFeatures,\n'
                                    '    MaximumPackingEfficiency,\n'
                                    '    StructuralComplexity,\n'
                                    ')\n'
                                    'from pymatgen.core import Composition\n'
                                    '\n'
                                    '\n'
                                    'FEATURE_SET_DISPLAY_NAMES = {\n'
                                    '    "magpie": "Magpie composition",\n'
                                    '    "magpie_structure": "Magpie + structure",\n'
                                    '}\n'
                                    '\n'
                                    '\n'
                                    'def load_matbench_task(task_name: str):\n'
                                    '    benchmark = MatbenchBenchmark(autoload=False, '
                                    'subset=[task_name])\n'
                                    '    task = list(benchmark.tasks)[0]\n'
                                    '    task.load()\n'
                                    '\n'
                                    '    if task.metadata["task_type"] != '
                                    '"regression":\n'
                                    '        raise ValueError(f"Expected regression '
                                    'task, got {task.metadata[\'task_type\']!r}.")\n'
                                    '    if task.metadata["input_type"] not in '
                                    '{"composition", "structure"}:\n'
                                    '        raise ValueError(f"Unsupported input '
                                    'type: {task.metadata[\'input_type\']!r}.")\n'
                                    '\n'
                                    '    return task\n'
                                    '\n'
                                    '\n'
                                    'def to_composition(value) -> Composition:\n'
                                    '    if isinstance(value, Composition):\n'
                                    '        return value\n'
                                    '    if hasattr(value, "composition"):\n'
                                    '        return value.composition\n'
                                    '    return Composition(value)\n'
                                    '\n'
                                    '\n'
                                    'def get_composition_inputs(task) -> '
                                    'tuple[pd.Series, str]:\n'
                                    '    """Return compositions for composition tasks '
                                    'and composition proxies for structures."""\n'
                                    '\n'
                                    '    input_type = task.metadata["input_type"]\n'
                                    '    if input_type == "composition":\n'
                                    '        return '
                                    'task.df[input_type].map(to_composition), '
                                    '"composition"\n'
                                    '    if input_type == "structure":\n'
                                    '        return task.df[input_type].map(lambda '
                                    'structure: structure.composition), (\n'
                                    '            "composition_from_structure"\n'
                                    '        )\n'
                                    '    raise ValueError(f"Unsupported input type: '
                                    '{input_type!r}.")\n'
                                    '\n'
                                    '\n'
                                    'def get_structure_inputs(task) -> '
                                    'tuple[pd.Series, str]:\n'
                                    '    input_type = task.metadata["input_type"]\n'
                                    '    if input_type != "structure":\n'
                                    '        raise ValueError(\n'
                                    '            f"Feature set requires structure '
                                    'input, got {input_type!r} for task."\n'
                                    '        )\n'
                                    '    return task.df[input_type], "structure"\n'
                                    '\n'
                                    '\n'
                                    'def _clean_feature_frame(features: pd.DataFrame) '
                                    '-> pd.DataFrame:\n'
                                    '    features = features.copy()\n'
                                    '    drop_columns = [\n'
                                    '        col\n'
                                    '        for col in features.columns\n'
                                    '        if col in {"composition", "structure"} or '
                                    'str(col).endswith(" Exceptions")\n'
                                    '    ]\n'
                                    '    if drop_columns:\n'
                                    '        features = '
                                    'features.drop(columns=drop_columns)\n'
                                    '\n'
                                    '    for col in '
                                    'features.select_dtypes(include=["bool"]).columns:\n'
                                    '        features[col] = '
                                    'features[col].astype(int)\n'
                                    '\n'
                                    '    categorical = '
                                    'features.select_dtypes(include=["object", '
                                    '"category"]).columns\n'
                                    '    if len(categorical):\n'
                                    '        features = pd.get_dummies(features, '
                                    'columns=list(categorical), dummy_na=True)\n'
                                    '\n'
                                    '    features = features.apply(pd.to_numeric, '
                                    'errors="coerce")\n'
                                    '    features = features.replace([np.inf, '
                                    '-np.inf], np.nan)\n'
                                    '    return features.dropna(axis=1, how="all")\n'
                                    '\n'
                                    '\n'
                                    'def featurize_magpie(\n'
                                    '    task_name: str,\n'
                                    '    compositions: pd.Series,\n'
                                    '    feature_dir: Path | str,\n'
                                    '    *,\n'
                                    '    use_cache: bool = True,\n'
                                    '    show_progress: bool = True,\n'
                                    '    n_jobs: int = 1,\n'
                                    ') -> pd.DataFrame:\n'
                                    '    """Build Magpie composition features with a '
                                    'run-local cache."""\n'
                                    '\n'
                                    '    feature_dir = Path(feature_dir)\n'
                                    '    feature_dir.mkdir(parents=True, '
                                    'exist_ok=True)\n'
                                    '    cache_path = feature_dir / '
                                    'f"{task_name}_magpie_features.csv"\n'
                                    '\n'
                                    '    if use_cache and cache_path.exists():\n'
                                    '        features = pd.read_csv(cache_path, '
                                    'index_col=0)\n'
                                    '        if len(features) != len(compositions):\n'
                                    '            raise ValueError(\n'
                                    '                f"Cached feature length mismatch '
                                    'for {task_name}: "\n'
                                    '                f"{len(features)} != '
                                    '{len(compositions)}."\n'
                                    '            )\n'
                                    '        features.index = compositions.index\n'
                                    '        return features\n'
                                    '\n'
                                    '    feature_input = pd.DataFrame({"composition": '
                                    'compositions.map(to_composition)})\n'
                                    '    feature_input.index = compositions.index\n'
                                    '\n'
                                    '    featurizer = '
                                    'ElementProperty.from_preset("magpie")\n'
                                    '    featurizer.set_n_jobs(n_jobs)\n'
                                    '    features = featurizer.featurize_dataframe(\n'
                                    '        feature_input,\n'
                                    '        col_id="composition",\n'
                                    '        ignore_errors=False,\n'
                                    '        inplace=False,\n'
                                    '        pbar=show_progress,\n'
                                    '    )\n'
                                    '    features = _clean_feature_frame(features)\n'
                                    '    features.to_csv(cache_path)\n'
                                    '    return features\n'
                                    '\n'
                                    '\n'
                                    'def featurize_structure_descriptors(\n'
                                    '    task_name: str,\n'
                                    '    structures: pd.Series,\n'
                                    '    feature_dir: Path | str,\n'
                                    '    *,\n'
                                    '    use_cache: bool = True,\n'
                                    '    show_progress: bool = True,\n'
                                    '    n_jobs: int = 1,\n'
                                    ') -> pd.DataFrame:\n'
                                    '    """Build lightweight structure descriptors '
                                    'for Matbench structure tasks."""\n'
                                    '\n'
                                    '    feature_dir = Path(feature_dir)\n'
                                    '    feature_dir.mkdir(parents=True, '
                                    'exist_ok=True)\n'
                                    '    cache_path = feature_dir / '
                                    'f"{task_name}_structure_features.csv"\n'
                                    '\n'
                                    '    if use_cache and cache_path.exists():\n'
                                    '        features = pd.read_csv(cache_path, '
                                    'index_col=0)\n'
                                    '        if len(features) != len(structures):\n'
                                    '            raise ValueError(\n'
                                    '                f"Cached structure feature length '
                                    'mismatch for {task_name}: "\n'
                                    '                f"{len(features)} != '
                                    '{len(structures)}."\n'
                                    '            )\n'
                                    '        features.index = structures.index\n'
                                    '        return features\n'
                                    '\n'
                                    '    features = '
                                    'pd.DataFrame(index=structures.index)\n'
                                    '    featurizers = [\n'
                                    '        DensityFeatures(),\n'
                                    '        GlobalSymmetryFeatures(),\n'
                                    '        MaximumPackingEfficiency(),\n'
                                    '        StructuralComplexity(),\n'
                                    '    ]\n'
                                    '\n'
                                    '    for featurizer in featurizers:\n'
                                    '        featurizer.set_n_jobs(n_jobs)\n'
                                    '        input_df = pd.DataFrame({"structure": '
                                    'structures}, index=structures.index)\n'
                                    '        output = featurizer.featurize_dataframe(\n'
                                    '            input_df,\n'
                                    '            col_id="structure",\n'
                                    '            ignore_errors=True,\n'
                                    '            return_errors=True,\n'
                                    '            inplace=False,\n'
                                    '            pbar=show_progress,\n'
                                    '        )\n'
                                    '        output = _clean_feature_frame(output)\n'
                                    '        prefix = type(featurizer).__name__\n'
                                    '        output = output.rename(columns=lambda '
                                    'col: f"{prefix}|{col}")\n'
                                    '        features = features.join(output, '
                                    'how="left")\n'
                                    '\n'
                                    '    features = _clean_feature_frame(features)\n'
                                    '    features.to_csv(cache_path)\n'
                                    '    return features\n'
                                    '\n'
                                    '\n'
                                    'def build_task_features(\n'
                                    '    task_name: str,\n'
                                    '    task,\n'
                                    '    feature_set: str,\n'
                                    '    feature_dir: Path | str,\n'
                                    '    *,\n'
                                    '    use_cache: bool = True,\n'
                                    '    show_progress: bool = True,\n'
                                    '    n_jobs: int = 1,\n'
                                    ') -> tuple[pd.DataFrame, str, str]:\n'
                                    '    """Return features, source label, and display '
                                    'label for one task/feature set."""\n'
                                    '\n'
                                    '    if feature_set == "magpie":\n'
                                    '        compositions, feature_source = '
                                    'get_composition_inputs(task)\n'
                                    '        features = featurize_magpie(\n'
                                    '            task_name,\n'
                                    '            compositions,\n'
                                    '            feature_dir,\n'
                                    '            use_cache=use_cache,\n'
                                    '            show_progress=show_progress,\n'
                                    '            n_jobs=n_jobs,\n'
                                    '        )\n'
                                    '        return features, feature_source, '
                                    'FEATURE_SET_DISPLAY_NAMES[feature_set]\n'
                                    '\n'
                                    '    if feature_set == "magpie_structure":\n'
                                    '        structures, feature_source = '
                                    'get_structure_inputs(task)\n'
                                    '        compositions = structures.map(lambda '
                                    'structure: structure.composition)\n'
                                    '        composition_features = featurize_magpie(\n'
                                    '            task_name,\n'
                                    '            compositions,\n'
                                    '            feature_dir,\n'
                                    '            use_cache=use_cache,\n'
                                    '            show_progress=show_progress,\n'
                                    '            n_jobs=n_jobs,\n'
                                    '        ).add_prefix("Magpie|")\n'
                                    '        structure_features = '
                                    'featurize_structure_descriptors(\n'
                                    '            task_name,\n'
                                    '            structures,\n'
                                    '            feature_dir,\n'
                                    '            use_cache=use_cache,\n'
                                    '            show_progress=show_progress,\n'
                                    '            n_jobs=n_jobs,\n'
                                    '        )\n'
                                    '        features = '
                                    'pd.concat([composition_features, '
                                    'structure_features], axis=1)\n'
                                    '        features = '
                                    '_clean_feature_frame(features)\n'
                                    '        return features, feature_source, '
                                    'FEATURE_SET_DISPLAY_NAMES[feature_set]\n'
                                    '\n'
                                    '    raise ValueError(f"Unsupported feature set: '
                                    '{feature_set!r}.")\n',
 'src/matbench_tabpfn/models.py': '"""Model builders used by the notebooks and '
                                  'scripts."""\n'
                                  '\n'
                                  'from __future__ import annotations\n'
                                  '\n'
                                  'from typing import Any\n'
                                  '\n'
                                  'import numpy as np\n'
                                  'import pandas as pd\n'
                                  'from sklearn.dummy import DummyRegressor\n'
                                  'from sklearn.ensemble import (\n'
                                  '    ExtraTreesRegressor,\n'
                                  '    HistGradientBoostingRegressor,\n'
                                  '    RandomForestRegressor,\n'
                                  ')\n'
                                  'from sklearn.impute import SimpleImputer\n'
                                  'from sklearn.linear_model import RidgeCV\n'
                                  'from sklearn.pipeline import Pipeline\n'
                                  'from sklearn.preprocessing import StandardScaler\n'
                                  'from tabpfn import TabPFNRegressor\n'
                                  '\n'
                                  'from .settings import RANDOM_SEED\n'
                                  '\n'
                                  '\n'
                                  'def build_model(\n'
                                  '    model_name: str,\n'
                                  '    *,\n'
                                  '    random_seed: int = RANDOM_SEED,\n'
                                  '    n_estimators: int = 500,\n'
                                  '    tabpfn_n_estimators: int = 8,\n'
                                  '    tabpfn_device: str = "cuda",\n'
                                  '    tabpfn_kwargs: dict[str, Any] | None = None,\n'
                                  ') -> Pipeline:\n'
                                  '    """Build one model with consistent '
                                  'preprocessing."""\n'
                                  '\n'
                                  '    if model_name == "dummy_mean":\n'
                                  '        steps = [\n'
                                  '            ("imputer", '
                                  'SimpleImputer(strategy="median")),\n'
                                  '            ("model", '
                                  'DummyRegressor(strategy="mean")),\n'
                                  '        ]\n'
                                  '    elif model_name == "ridge_cv":\n'
                                  '        steps = [\n'
                                  '            ("imputer", '
                                  'SimpleImputer(strategy="median")),\n'
                                  '            ("scaler", StandardScaler()),\n'
                                  '            ("model", '
                                  'RidgeCV(alphas=np.logspace(-6, 6, 25))),\n'
                                  '        ]\n'
                                  '    elif model_name == "random_forest":\n'
                                  '        steps = [\n'
                                  '            ("imputer", '
                                  'SimpleImputer(strategy="median")),\n'
                                  '            (\n'
                                  '                "model",\n'
                                  '                RandomForestRegressor(\n'
                                  '                    n_estimators=n_estimators,\n'
                                  '                    random_state=random_seed,\n'
                                  '                    n_jobs=-1,\n'
                                  '                ),\n'
                                  '            ),\n'
                                  '        ]\n'
                                  '    elif model_name == "extra_trees":\n'
                                  '        steps = [\n'
                                  '            ("imputer", '
                                  'SimpleImputer(strategy="median")),\n'
                                  '            (\n'
                                  '                "model",\n'
                                  '                ExtraTreesRegressor(\n'
                                  '                    n_estimators=n_estimators,\n'
                                  '                    random_state=random_seed,\n'
                                  '                    n_jobs=-1,\n'
                                  '                ),\n'
                                  '            ),\n'
                                  '        ]\n'
                                  '    elif model_name == "hist_gradient_boosting":\n'
                                  '        steps = [\n'
                                  '            ("imputer", '
                                  'SimpleImputer(strategy="median")),\n'
                                  '            (\n'
                                  '                "model",\n'
                                  '                HistGradientBoostingRegressor(\n'
                                  '                    max_iter=500,\n'
                                  '                    learning_rate=0.04,\n'
                                  '                    l2_regularization=0.01,\n'
                                  '                    early_stopping=True,\n'
                                  '                    random_state=random_seed,\n'
                                  '                ),\n'
                                  '            ),\n'
                                  '        ]\n'
                                  '    elif model_name == "tabpfn":\n'
                                  '        kwargs = {\n'
                                  '            "n_estimators": tabpfn_n_estimators,\n'
                                  '            "random_state": random_seed,\n'
                                  '            "device": tabpfn_device,\n'
                                  '            "inference_precision": "auto",\n'
                                  '            "memory_saving_mode": "auto",\n'
                                  '            "show_progress_bar": False,\n'
                                  '        }\n'
                                  '        if tabpfn_kwargs:\n'
                                  '            kwargs.update(tabpfn_kwargs)\n'
                                  '        steps = [\n'
                                  '            ("imputer", '
                                  'SimpleImputer(strategy="median")),\n'
                                  '            ("model", TabPFNRegressor(**kwargs)),\n'
                                  '        ]\n'
                                  '    else:\n'
                                  '        raise ValueError(f"Unsupported model: '
                                  '{model_name!r}.")\n'
                                  '\n'
                                  '    return Pipeline(steps)\n'
                                  '\n'
                                  '\n'
                                  'def predict_in_batches(\n'
                                  '    model: Pipeline,\n'
                                  '    X_test: pd.DataFrame,\n'
                                  '    batch_size: int | None,\n'
                                  ') -> np.ndarray:\n'
                                  '    """Predict with optional final-estimator '
                                  'batching for GPU memory control."""\n'
                                  '\n'
                                  '    if batch_size is None or batch_size <= 0 or '
                                  'batch_size >= len(X_test):\n'
                                  '        return model.predict(X_test)\n'
                                  '\n'
                                  '    preprocessor = model[:-1]\n'
                                  '    regressor = model.steps[-1][1]\n'
                                  '    X_test_preprocessed = '
                                  'preprocessor.transform(X_test)\n'
                                  '\n'
                                  '    predictions = []\n'
                                  '    for start in range(0, len(X_test_preprocessed), '
                                  'batch_size):\n'
                                  '        stop = start + batch_size\n'
                                  '        '
                                  'predictions.append(regressor.predict(X_test_preprocessed[start:stop]))\n'
                                  '    return np.concatenate(predictions)\n',
 'src/matbench_tabpfn/paths.py': '"""Output path helpers for reproducible experiment '
                                 'runs."""\n'
                                 '\n'
                                 'from __future__ import annotations\n'
                                 '\n'
                                 'import json\n'
                                 'import os\n'
                                 'import platform\n'
                                 'import shutil\n'
                                 'import sys\n'
                                 'from dataclasses import asdict, dataclass\n'
                                 'from datetime import datetime, timezone\n'
                                 'from pathlib import Path\n'
                                 'from typing import Any\n'
                                 '\n'
                                 'import numpy as np\n'
                                 'import pandas as pd\n'
                                 '\n'
                                 'from .settings import PROJECT_ROOT\n'
                                 '\n'
                                 '\n'
                                 '@dataclass(frozen=True)\n'
                                 'class RunPaths:\n'
                                 '    """Canonical paths for a single experiment '
                                 'run."""\n'
                                 '\n'
                                 '    root: Path\n'
                                 '    metrics: Path\n'
                                 '    predictions: Path\n'
                                 '    features: Path\n'
                                 '    figures: Path\n'
                                 '    tables: Path\n'
                                 '    logs: Path\n'
                                 '\n'
                                 '    def mkdirs(self) -> None:\n'
                                 '        for path in asdict(self).values():\n'
                                 '            Path(path).mkdir(parents=True, '
                                 'exist_ok=True)\n'
                                 '\n'
                                 '    def as_posix_dict(self) -> dict[str, str]:\n'
                                 '        return {key: str(value) for key, value in '
                                 'asdict(self).items()}\n'
                                 '\n'
                                 '\n'
                                 'def default_run_id(prefix: str = "gpu") -> str:\n'
                                 '    stamp = '
                                 'datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")\n'
                                 '    return f"{prefix}_{stamp}"\n'
                                 '\n'
                                 '\n'
                                 'def create_run_paths(\n'
                                 '    project_root: Path | str = PROJECT_ROOT,\n'
                                 '    *,\n'
                                 '    run_id: str | None = None,\n'
                                 '    base_dir: Path | str = "results/runs",\n'
                                 '    latest_root: Path | str | None = None,\n'
                                 '    clean: bool = False,\n'
                                 '    update_latest: bool = True,\n'
                                 ') -> RunPaths:\n'
                                 '    """Create a unified output tree for one run."""\n'
                                 '\n'
                                 '    project_root = Path(project_root).resolve()\n'
                                 '    run_id = run_id or default_run_id()\n'
                                 '    base_path = Path(base_dir)\n'
                                 '    if not base_path.is_absolute():\n'
                                 '        base_path = project_root / base_path\n'
                                 '\n'
                                 '    root = base_path / run_id\n'
                                 '    if root.exists() and clean:\n'
                                 '        shutil.rmtree(root)\n'
                                 '    root.mkdir(parents=True, exist_ok=True)\n'
                                 '\n'
                                 '    paths = RunPaths(\n'
                                 '        root=root,\n'
                                 '        metrics=root / "metrics",\n'
                                 '        predictions=root / "predictions",\n'
                                 '        features=root / "features",\n'
                                 '        figures=root / "figures",\n'
                                 '        tables=root / "tables",\n'
                                 '        logs=root / "logs",\n'
                                 '    )\n'
                                 '    paths.mkdirs()\n'
                                 '\n'
                                 '    if update_latest:\n'
                                 '        latest_parent = Path(latest_root).resolve() '
                                 'if latest_root else project_root / "results"\n'
                                 '        latest_parent.mkdir(parents=True, '
                                 'exist_ok=True)\n'
                                 '        latest = latest_parent / "latest"\n'
                                 '        if latest.exists() or latest.is_symlink():\n'
                                 '            if latest.is_symlink() or '
                                 'latest.is_file():\n'
                                 '                latest.unlink()\n'
                                 '            else:\n'
                                 '                shutil.rmtree(latest)\n'
                                 '        try:\n'
                                 '            latest.symlink_to(root, '
                                 'target_is_directory=True)\n'
                                 '        except OSError:\n'
                                 '            (latest_parent / '
                                 '"latest_run.txt").write_text(\n'
                                 '                str(root) + "\\n", encoding="utf-8"\n'
                                 '            )\n'
                                 '\n'
                                 '    return paths\n'
                                 '\n'
                                 '\n'
                                 'def _json_safe(value: Any) -> Any:\n'
                                 '    if isinstance(value, Path):\n'
                                 '        return str(value)\n'
                                 '    if isinstance(value, np.generic):\n'
                                 '        return value.item()\n'
                                 '    if isinstance(value, dict):\n'
                                 '        return {str(key): _json_safe(val) for key, '
                                 'val in value.items()}\n'
                                 '    if isinstance(value, (list, tuple)):\n'
                                 '        return [_json_safe(item) for item in value]\n'
                                 '    return value\n'
                                 '\n'
                                 '\n'
                                 'def write_json(path: Path | str, data: dict[str, '
                                 'Any]) -> None:\n'
                                 '    Path(path).write_text(\n'
                                 '        json.dumps(_json_safe(data), indent=2, '
                                 'sort_keys=True) + "\\n",\n'
                                 '        encoding="utf-8",\n'
                                 '    )\n'
                                 '\n'
                                 '\n'
                                 'def collect_environment_manifest(extra: dict[str, '
                                 'Any] | None = None) -> dict[str, Any]:\n'
                                 '    """Collect enough environment details to make a '
                                 'run auditable."""\n'
                                 '\n'
                                 '    manifest: dict[str, Any] = {\n'
                                 '        "created_at_utc": '
                                 'datetime.now(timezone.utc).isoformat(),\n'
                                 '        "python": sys.version,\n'
                                 '        "python_executable": sys.executable,\n'
                                 '        "platform": platform.platform(),\n'
                                 '        "cwd": os.getcwd(),\n'
                                 '        "packages": {\n'
                                 '            "numpy": np.__version__,\n'
                                 '            "pandas": pd.__version__,\n'
                                 '        },\n'
                                 '    }\n'
                                 '\n'
                                 '    try:\n'
                                 '        import sklearn\n'
                                 '\n'
                                 '        manifest["packages"]["scikit_learn"] = '
                                 'sklearn.__version__\n'
                                 '    except Exception as exc:  # pragma: no cover - '
                                 'manifest best effort\n'
                                 '        manifest["packages"]["scikit_learn_error"] = '
                                 'repr(exc)\n'
                                 '\n'
                                 '    try:\n'
                                 '        import matbench\n'
                                 '\n'
                                 '        manifest["packages"]["matbench"] = '
                                 'getattr(matbench, "__version__", "unknown")\n'
                                 '    except Exception as exc:  # pragma: no cover - '
                                 'manifest best effort\n'
                                 '        manifest["packages"]["matbench_error"] = '
                                 'repr(exc)\n'
                                 '\n'
                                 '    try:\n'
                                 '        import matminer\n'
                                 '\n'
                                 '        manifest["packages"]["matminer"] = '
                                 'getattr(matminer, "__version__", "unknown")\n'
                                 '    except Exception as exc:  # pragma: no cover - '
                                 'manifest best effort\n'
                                 '        manifest["packages"]["matminer_error"] = '
                                 'repr(exc)\n'
                                 '\n'
                                 '    try:\n'
                                 '        import tabpfn\n'
                                 '\n'
                                 '        manifest["packages"]["tabpfn"] = '
                                 'getattr(tabpfn, "__version__", "unknown")\n'
                                 '    except Exception as exc:  # pragma: no cover - '
                                 'manifest best effort\n'
                                 '        manifest["packages"]["tabpfn_error"] = '
                                 'repr(exc)\n'
                                 '\n'
                                 '    try:\n'
                                 '        import torch\n'
                                 '\n'
                                 '        manifest["packages"]["torch"] = '
                                 'torch.__version__\n'
                                 '        manifest["cuda"] = {\n'
                                 '            "available": torch.cuda.is_available(),\n'
                                 '            "device_count": '
                                 'torch.cuda.device_count(),\n'
                                 '            "devices": [\n'
                                 '                {\n'
                                 '                    "index": idx,\n'
                                 '                    "name": '
                                 'torch.cuda.get_device_name(idx),\n'
                                 '                    "total_memory_gb": round(\n'
                                 '                        '
                                 'torch.cuda.get_device_properties(idx).total_memory / '
                                 '1024**3, 3\n'
                                 '                    ),\n'
                                 '                }\n'
                                 '                for idx in '
                                 'range(torch.cuda.device_count())\n'
                                 '            ],\n'
                                 '        }\n'
                                 '    except Exception as exc:  # pragma: no cover - '
                                 'manifest best effort\n'
                                 '        manifest["cuda"] = {"error": repr(exc)}\n'
                                 '\n'
                                 '    if extra:\n'
                                 '        manifest["config"] = extra\n'
                                 '\n'
                                 '    return manifest\n',
 'src/matbench_tabpfn/plotting.py': '"""Unified plotting style and report figures."""\n'
                                    '\n'
                                    'from __future__ import annotations\n'
                                    '\n'
                                    'import math\n'
                                    'from pathlib import Path\n'
                                    '\n'
                                    'import matplotlib.pyplot as plt\n'
                                    'import numpy as np\n'
                                    'import pandas as pd\n'
                                    'import seaborn as sns\n'
                                    '\n'
                                    'from .paths import RunPaths\n'
                                    'from .settings import MODEL_COLORS, '
                                    'MODEL_DISPLAY_NAMES, DEFAULT_MODELS\n'
                                    '\n'
                                    '\n'
                                    'PLOT_STYLE_VERSION = "robust_fold_axis_v1"\n'
                                    '\n'
                                    '\n'
                                    'def set_plot_style() -> None:\n'
                                    '    sns.set_theme(\n'
                                    '        context="paper",\n'
                                    '        style="whitegrid",\n'
                                    '        font="DejaVu Sans",\n'
                                    '        rc={\n'
                                    '            "figure.dpi": 120,\n'
                                    '            "savefig.dpi": 220,\n'
                                    '            "axes.spines.top": False,\n'
                                    '            "axes.spines.right": False,\n'
                                    '            "axes.titleweight": "bold",\n'
                                    '            "axes.labelsize": 10,\n'
                                    '            "axes.titlesize": 11,\n'
                                    '            "xtick.labelsize": 9,\n'
                                    '            "ytick.labelsize": 9,\n'
                                    '            "legend.fontsize": 9,\n'
                                    '            "legend.title_fontsize": 9,\n'
                                    '            "grid.alpha": 0.25,\n'
                                    '        },\n'
                                    '    )\n'
                                    '\n'
                                    '\n'
                                    'def _save(fig: plt.Figure, path: Path) -> Path:\n'
                                    '    path.parent.mkdir(parents=True, '
                                    'exist_ok=True)\n'
                                    '    fig.tight_layout(rect=(0, 0, 1, 0.98))\n'
                                    '    fig.savefig(path, bbox_inches="tight")\n'
                                    '    fig.savefig(path.with_suffix(".pdf"), '
                                    'bbox_inches="tight")\n'
                                    '    plt.close(fig)\n'
                                    '    return path\n'
                                    '\n'
                                    '\n'
                                    'def _ordered_models(models: pd.Series) -> '
                                    'list[str]:\n'
                                    '    present = '
                                    'list(dict.fromkeys(models.dropna().tolist()))\n'
                                    '    ordered = [model for model in DEFAULT_MODELS '
                                    'if model in present]\n'
                                    '    ordered.extend([model for model in present if '
                                    'model not in ordered])\n'
                                    '    return ordered\n'
                                    '\n'
                                    '\n'
                                    'def _add_plot_labels(df: pd.DataFrame) -> '
                                    'pd.DataFrame:\n'
                                    '    df = df.copy()\n'
                                    '    if "feature_set_display" not in df.columns:\n'
                                    '        df["feature_set_display"] = '
                                    'df.get("feature_set", "")\n'
                                    '    df["plot_label"] = '
                                    'df["model_display"].astype(str)\n'
                                    '    for task_name, task_df in '
                                    'df.groupby("task"):\n'
                                    '        if task_df["feature_set"].nunique() > 1:\n'
                                    '            mask = df["task"] == task_name\n'
                                    '            feature_label = df.loc[mask, '
                                    '"feature_set"].map(\n'
                                    '                {\n'
                                    '                    "magpie": "comp",\n'
                                    '                    "magpie_structure": '
                                    '"struct",\n'
                                    '                }\n'
                                    '            )\n'
                                    '            feature_label = '
                                    'feature_label.fillna(df.loc[mask, '
                                    '"feature_set"].astype(str))\n'
                                    '            df.loc[mask, "plot_label"] = (\n'
                                    '                df.loc[mask, '
                                    '"model_display"].astype(str) + " | " + '
                                    'feature_label\n'
                                    '            )\n'
                                    '    return df\n'
                                    '\n'
                                    '\n'
                                    'def _short_feature_label(feature_set: str, '
                                    'feature_set_display: str | None = None) -> str:\n'
                                    '    labels = {\n'
                                    '        "magpie": "comp",\n'
                                    '        "magpie_structure": "struct",\n'
                                    '    }\n'
                                    '    return labels.get(feature_set, '
                                    'feature_set_display or feature_set)\n'
                                    '\n'
                                    '\n'
                                    'def _robust_fold_axis_upper(values: pd.Series) -> '
                                    'tuple[float, bool]:\n'
                                    '    """Return a plot-only y-axis cap that keeps '
                                    'single catastrophic folds visible."""\n'
                                    '    clean = pd.to_numeric(pd.Series(values), '
                                    'errors="coerce").dropna()\n'
                                    '    if clean.empty:\n'
                                    '        return 1.0, False\n'
                                    '\n'
                                    '    maximum = float(clean.max())\n'
                                    '    if maximum <= 0:\n'
                                    '        return 1.0, False\n'
                                    '    if len(clean) < 8:\n'
                                    '        return maximum * 1.12, False\n'
                                    '\n'
                                    '    q1, q3 = np.percentile(clean, [25, 75])\n'
                                    '    sorted_values = np.sort(clean.to_numpy())\n'
                                    '    p90 = sorted_values[int(math.floor(0.90 * '
                                    '(len(sorted_values) - 1)))]\n'
                                    '    iqr = q3 - q1\n'
                                    '    whisker_top = q3 + 1.5 * iqr if iqr > 0 else '
                                    'p90\n'
                                    '    typical_top = max(float(p90), '
                                    'float(whisker_top))\n'
                                    '    if typical_top <= 0:\n'
                                    '        return maximum * 1.12, False\n'
                                    '\n'
                                    '    should_clip = maximum > typical_top * 3.0 and '
                                    '(maximum - typical_top) > 0.2 * maximum\n'
                                    '    if should_clip:\n'
                                    '        return typical_top * 1.18, True\n'
                                    '    return maximum * 1.12, False\n'
                                    '\n'
                                    '\n'
                                    'def _grid_for_tasks(\n'
                                    '    tasks: list[str],\n'
                                    '    row_counts: dict[str, int],\n'
                                    '    *,\n'
                                    '    width_per_col: float,\n'
                                    '    min_panel_height: float,\n'
                                    '    height_per_item: float,\n'
                                    ') -> tuple[plt.Figure, np.ndarray]:\n'
                                    '    ncols = min(2, len(tasks))\n'
                                    '    nrows = math.ceil(len(tasks) / ncols)\n'
                                    '    max_items = max(row_counts.values()) if '
                                    'row_counts else 1\n'
                                    '    panel_height = max(min_panel_height, 1.1 + '
                                    'height_per_item * max_items)\n'
                                    '    fig, axes = plt.subplots(\n'
                                    '        nrows,\n'
                                    '        ncols,\n'
                                    '        figsize=(width_per_col * ncols, '
                                    'panel_height * nrows),\n'
                                    '        squeeze=False,\n'
                                    '    )\n'
                                    '    return fig, axes.reshape(-1)\n'
                                    '\n'
                                    '\n'
                                    'def plot_model_mae_comparison(summary_df: '
                                    'pd.DataFrame, figures_dir: Path) -> Path | None:\n'
                                    '    if summary_df.empty:\n'
                                    '        return None\n'
                                    '\n'
                                    '    set_plot_style()\n'
                                    '    tasks = '
                                    'list(summary_df["task"].drop_duplicates())\n'
                                    '    row_counts = {\n'
                                    '        task_name: len(summary_df.query("task == '
                                    '@task_name")) for task_name in tasks\n'
                                    '    }\n'
                                    '    fig, axes = _grid_for_tasks(\n'
                                    '        tasks,\n'
                                    '        row_counts,\n'
                                    '        width_per_col=8.5,\n'
                                    '        min_panel_height=3.6,\n'
                                    '        height_per_item=0.43,\n'
                                    '    )\n'
                                    '\n'
                                    '    for ax, task_name in zip(axes, tasks):\n'
                                    '        task_summary = _add_plot_labels(\n'
                                    '            summary_df.query("task == '
                                    '@task_name").sort_values("mean_mae")\n'
                                    '        )\n'
                                    '        y = np.arange(len(task_summary))\n'
                                    '        colors = [MODEL_COLORS.get(model, '
                                    '"#777777") for model in task_summary["model"]]\n'
                                    '        ax.barh(\n'
                                    '            y,\n'
                                    '            task_summary["mean_mae"],\n'
                                    '            '
                                    'xerr=task_summary["sem_mae"].fillna(0),\n'
                                    '            color=colors,\n'
                                    '            alpha=0.88,\n'
                                    '        )\n'
                                    '        ax.set_yticks(y)\n'
                                    '        '
                                    'ax.set_yticklabels(task_summary["plot_label"])\n'
                                    '        ax.invert_yaxis()\n'
                                    '        unit = task_summary["unit"].iloc[0]\n'
                                    '        ax.set_title(task_name)\n'
                                    '        ax.set_xlabel(f"Mean MAE ({unit})")\n'
                                    '        xmax = (\n'
                                    '            task_summary["mean_mae"] + '
                                    'task_summary["sem_mae"].fillna(0)\n'
                                    '        ).max()\n'
                                    '        ax.set_xlim(left=0, right=xmax * 1.18 if '
                                    'xmax > 0 else 1)\n'
                                    '        for idx, value in '
                                    'enumerate(task_summary["mean_mae"]):\n'
                                    '            ax.text(value, idx, f" {value:.3g}", '
                                    'va="center", fontsize=8)\n'
                                    '        ax.tick_params(axis="y", labelsize=8)\n'
                                    '\n'
                                    '    for ax in axes[len(tasks) :]:\n'
                                    '        ax.axis("off")\n'
                                    '\n'
                                    '    fig.suptitle("Official-fold MAE by task and '
                                    'model", y=0.995, fontsize=13)\n'
                                    '    return _save(fig, figures_dir / '
                                    '"01_model_mae_comparison.png")\n'
                                    '\n'
                                    '\n'
                                    'def plot_fold_mae_distribution(metrics_df: '
                                    'pd.DataFrame, figures_dir: Path) -> Path | None:\n'
                                    '    ok = metrics_df.query("status == '
                                    '\'ok\'").copy()\n'
                                    '    if ok.empty:\n'
                                    '        return None\n'
                                    '\n'
                                    '    set_plot_style()\n'
                                    '    tasks = list(ok["task"].drop_duplicates())\n'
                                    '    row_counts = {\n'
                                    '        task_name: ok.query("task == '
                                    '@task_name")["model"].nunique()\n'
                                    '        * ok.query("task == '
                                    '@task_name")["feature_set"].nunique()\n'
                                    '        for task_name in tasks\n'
                                    '    }\n'
                                    '    fig, axes = _grid_for_tasks(\n'
                                    '        tasks,\n'
                                    '        row_counts,\n'
                                    '        width_per_col=8.8,\n'
                                    '        min_panel_height=4.0,\n'
                                    '        height_per_item=0.2,\n'
                                    '    )\n'
                                    '\n'
                                    '    for ax, task_name in zip(axes, tasks):\n'
                                    '        task_metrics = '
                                    '_add_plot_labels(ok.query("task == '
                                    '@task_name").copy())\n'
                                    '        if task_metrics["feature_set"].nunique() '
                                    '> 1:\n'
                                    '            display_order = (\n'
                                    '                '
                                    'task_metrics.groupby("plot_label")["mae"]\n'
                                    '                .mean()\n'
                                    '                .sort_values()\n'
                                    '                .index.tolist()\n'
                                    '            )\n'
                                    '        else:\n'
                                    '            order = '
                                    '_ordered_models(task_metrics["model"])\n'
                                    '            display_order = '
                                    '[MODEL_DISPLAY_NAMES.get(model, model) for model '
                                    'in order]\n'
                                    '        sns.boxplot(\n'
                                    '            data=task_metrics,\n'
                                    '            x="plot_label",\n'
                                    '            y="mae",\n'
                                    '            order=display_order,\n'
                                    '            ax=ax,\n'
                                    '            width=0.55,\n'
                                    '            color="#d9dee7",\n'
                                    '            fliersize=0,\n'
                                    '        )\n'
                                    '        sns.stripplot(\n'
                                    '            data=task_metrics,\n'
                                    '            x="plot_label",\n'
                                    '            y="mae",\n'
                                    '            order=display_order,\n'
                                    '            hue="model",\n'
                                    '            palette=MODEL_COLORS,\n'
                                    '            ax=ax,\n'
                                    '            size=4,\n'
                                    '            jitter=0.18,\n'
                                    '            legend=False,\n'
                                    '        )\n'
                                    '        unit = task_metrics["unit"].iloc[0]\n'
                                    '        ax.set_title(task_name)\n'
                                    '        ax.set_xlabel("")\n'
                                    '        ax.set_ylabel(f"Fold MAE ({unit})")\n'
                                    '        axis_upper, has_clipped = '
                                    '_robust_fold_axis_upper(task_metrics["mae"])\n'
                                    '        ax.set_ylim(bottom=0, top=axis_upper)\n'
                                    '        if has_clipped:\n'
                                    '            clipped = task_metrics.query("mae > '
                                    '@axis_upper").copy()\n'
                                    '            if not clipped.empty:\n'
                                    '                label_to_x = {label: idx for idx, '
                                    'label in enumerate(display_order)}\n'
                                    '                within_label = '
                                    'clipped.groupby("plot_label").cumcount().astype(float)\n'
                                    '                label_count = (\n'
                                    '                    '
                                    'clipped.groupby("plot_label")["plot_label"]\n'
                                    '                    .transform("size")\n'
                                    '                    .astype(float)\n'
                                    '                )\n'
                                    '                x_positions = (\n'
                                    '                    '
                                    'clipped["plot_label"].map(label_to_x).astype(float)\n'
                                    '                    + (within_label - '
                                    '(label_count - 1) / 2) * 0.08\n'
                                    '                )\n'
                                    '                y_positions = '
                                    'np.full(len(clipped), axis_upper * 0.985)\n'
                                    '                colors = [MODEL_COLORS.get(model, '
                                    '"#777777") for model in clipped["model"]]\n'
                                    '                ax.scatter(\n'
                                    '                    x_positions,\n'
                                    '                    y_positions,\n'
                                    '                    marker="^",\n'
                                    '                    s=50,\n'
                                    '                    color=colors,\n'
                                    '                    edgecolors="white",\n'
                                    '                    linewidth=0.7,\n'
                                    '                    zorder=5,\n'
                                    '                )\n'
                                    '                noun = "fold" if len(clipped) == '
                                    '1 else "folds"\n'
                                    '                ax.text(\n'
                                    '                    0.98,\n'
                                    '                    0.95,\n'
                                    '                    f"{len(clipped)} clipped '
                                    "{noun}; max={clipped['mae'].max():.3g} "
                                    '{unit}",\n'
                                    '                    transform=ax.transAxes,\n'
                                    '                    ha="right",\n'
                                    '                    va="top",\n'
                                    '                    fontsize=8,\n'
                                    '                    bbox={\n'
                                    '                        "boxstyle": '
                                    '"round,pad=0.25",\n'
                                    '                        "facecolor": "white",\n'
                                    '                        "edgecolor": "#c8cdd4",\n'
                                    '                        "alpha": 0.86,\n'
                                    '                    },\n'
                                    '                )\n'
                                    '        ax.tick_params(axis="x", rotation=35, '
                                    'labelsize=8)\n'
                                    '        for label in ax.get_xticklabels():\n'
                                    '            label.set_ha("right")\n'
                                    '\n'
                                    '    for ax in axes[len(tasks) :]:\n'
                                    '        ax.axis("off")\n'
                                    '\n'
                                    '    fig.suptitle("Fold-level MAE dispersion", '
                                    'y=0.995, fontsize=13)\n'
                                    '    return _save(fig, figures_dir / '
                                    '"02_fold_mae_distribution.png")\n'
                                    '\n'
                                    '\n'
                                    'def plot_tabpfn_vs_baseline(comparison_df: '
                                    'pd.DataFrame, figures_dir: Path) -> Path | None:\n'
                                    '    if comparison_df.empty or "tabpfn" not in '
                                    'set(comparison_df["model"]):\n'
                                    '        return None\n'
                                    '\n'
                                    '    tabpfn = comparison_df.query("model == '
                                    '\'tabpfn\'").copy()\n'
                                    '    if tabpfn.empty:\n'
                                    '        return None\n'
                                    '    tabpfn = _add_plot_labels(tabpfn)\n'
                                    '    tabpfn["x_label"] = tabpfn["task"]\n'
                                    '    if '
                                    'tabpfn.groupby("task")["feature_set"].transform("nunique").max() '
                                    '> 1:\n'
                                    '        tabpfn["x_label"] = tabpfn["task"] + '
                                    '"\\n" + tabpfn["feature_set_display"]\n'
                                    '\n'
                                    '    set_plot_style()\n'
                                    '    fig, ax = plt.subplots(figsize=(max(8.2, 1.4 '
                                    '* len(tabpfn)), 3.8))\n'
                                    '    colors = [\n'
                                    '        "#3f8f5f" if value < 0 else "#b44c4c"\n'
                                    '        for value in '
                                    'tabpfn["mae_pct_change_vs_best_baseline"]\n'
                                    '    ]\n'
                                    '    ax.bar(\n'
                                    '        tabpfn["x_label"],\n'
                                    '        '
                                    'tabpfn["mae_pct_change_vs_best_baseline"],\n'
                                    '        color=colors,\n'
                                    '        alpha=0.9,\n'
                                    '    )\n'
                                    '    ax.axhline(0, color="#333333", linewidth=1)\n'
                                    '    ax.set_ylabel("MAE change vs best non-TabPFN '
                                    'baseline (%)")\n'
                                    '    ax.set_xlabel("")\n'
                                    '    ax.set_title("TabPFN relative to the best '
                                    'classical baseline")\n'
                                    '    ax.tick_params(axis="x", rotation=20)\n'
                                    '    for idx, row in '
                                    'tabpfn.reset_index(drop=True).iterrows():\n'
                                    '        value = '
                                    'row["mae_pct_change_vs_best_baseline"]\n'
                                    '        va = "bottom" if value >= 0 else "top"\n'
                                    '        ax.text(idx, value, f"{value:+.1f}%", '
                                    'ha="center", va=va, fontsize=9)\n'
                                    '    return _save(fig, figures_dir / '
                                    '"03_tabpfn_vs_best_baseline.png")\n'
                                    '\n'
                                    '\n'
                                    'def plot_tabpfn_parity(predictions_df: '
                                    'pd.DataFrame, figures_dir: Path) -> Path | None:\n'
                                    '    if predictions_df.empty or "tabpfn" not in '
                                    'set(predictions_df["model"]):\n'
                                    '        return None\n'
                                    '\n'
                                    '    data = predictions_df.query("model == '
                                    '\'tabpfn\'").copy()\n'
                                    '    if data.empty:\n'
                                    '        return None\n'
                                    '\n'
                                    '    set_plot_style()\n'
                                    '    groups = list(data.groupby(["task", '
                                    '"feature_set"], sort=False))\n'
                                    '    ncols = min(2, len(groups))\n'
                                    '    nrows = math.ceil(len(groups) / ncols)\n'
                                    '    fig, axes = plt.subplots(nrows, ncols, '
                                    'figsize=(5.2 * ncols, 4.6 * nrows))\n'
                                    '    axes = np.atleast_1d(axes).reshape(-1)\n'
                                    '\n'
                                    '    for ax, ((task_name, feature_set), task_pred) '
                                    'in zip(axes, groups):\n'
                                    '        ax.scatter(\n'
                                    '            task_pred["y_true"],\n'
                                    '            task_pred["y_pred"],\n'
                                    '            s=18,\n'
                                    '            alpha=0.55,\n'
                                    '            color=MODEL_COLORS["tabpfn"],\n'
                                    '            edgecolors="none",\n'
                                    '        )\n'
                                    '        low = min(task_pred["y_true"].min(), '
                                    'task_pred["y_pred"].min())\n'
                                    '        high = max(task_pred["y_true"].max(), '
                                    'task_pred["y_pred"].max())\n'
                                    '        pad = 0.04 * (high - low) if high > low '
                                    'else 1.0\n'
                                    '        low -= pad\n'
                                    '        high += pad\n'
                                    '        ax.plot([low, high], [low, high], '
                                    'color="#333333", linewidth=1, linestyle="--")\n'
                                    '        ax.set_xlim(low, high)\n'
                                    '        ax.set_ylim(low, high)\n'
                                    '        unit = task_pred["unit"].iloc[0]\n'
                                    '        mae = task_pred["absolute_error"].mean()\n'
                                    '        feature_label = '
                                    'task_pred["feature_set_display"].iloc[0]\n'
                                    '        short_feature = '
                                    '_short_feature_label(feature_set, feature_label)\n'
                                    '        ax.set_title(task_name, fontsize=11, '
                                    'pad=8)\n'
                                    '        ax.text(\n'
                                    '            0.03,\n'
                                    '            0.97,\n'
                                    '            f"{short_feature} | MAE={mae:.3g} '
                                    '{unit}",\n'
                                    '            transform=ax.transAxes,\n'
                                    '            va="top",\n'
                                    '            ha="left",\n'
                                    '            fontsize=9,\n'
                                    '            bbox={\n'
                                    '                "boxstyle": "round,pad=0.25",\n'
                                    '                "facecolor": "white",\n'
                                    '                "edgecolor": "none",\n'
                                    '                "alpha": 0.78,\n'
                                    '            },\n'
                                    '        )\n'
                                    '        ax.set_xlabel(f"True ({unit})")\n'
                                    '        ax.set_ylabel(f"Predicted ({unit})")\n'
                                    '\n'
                                    '    for ax in axes[len(groups) :]:\n'
                                    '        ax.axis("off")\n'
                                    '\n'
                                    '    fig.suptitle("TabPFN parity plots across '
                                    'official folds", y=1.02, fontsize=13)\n'
                                    '    return _save(fig, figures_dir / '
                                    '"04_tabpfn_parity.png")\n'
                                    '\n'
                                    '\n'
                                    'def plot_structure_feature_set_comparison(\n'
                                    '    summary_df: pd.DataFrame, figures_dir: Path\n'
                                    ') -> Path | None:\n'
                                    '    if summary_df.empty or "feature_set" not in '
                                    'summary_df.columns:\n'
                                    '        return None\n'
                                    '\n'
                                    '    data = summary_df.query("matbench_input_type '
                                    '== \'structure\'").copy()\n'
                                    '    if data.empty or '
                                    'data["feature_set"].nunique() < 2:\n'
                                    '        return None\n'
                                    '\n'
                                    '    best_by_feature = (\n'
                                    '        data.sort_values("mean_mae")\n'
                                    '        .groupby(["task", "feature_set", '
                                    '"feature_set_display"], as_index=False)\n'
                                    '        .first()\n'
                                    '    )\n'
                                    '\n'
                                    '    set_plot_style()\n'
                                    '    tasks = '
                                    'list(best_by_feature["task"].drop_duplicates())\n'
                                    '    ncols = min(2, len(tasks))\n'
                                    '    nrows = math.ceil(len(tasks) / ncols)\n'
                                    '    fig, axes = plt.subplots(nrows, ncols, '
                                    'figsize=(6.4 * ncols, 3.4 * nrows))\n'
                                    '    axes = np.atleast_1d(axes).reshape(-1)\n'
                                    '\n'
                                    '    for ax, task_name in zip(axes, tasks):\n'
                                    '        task_df = best_by_feature.query("task == '
                                    '@task_name").sort_values("mean_mae")\n'
                                    '        labels = task_df["feature_set_display"]\n'
                                    '        ax.barh(labels, task_df["mean_mae"], '
                                    'color="#607d8b", alpha=0.88)\n'
                                    '        ax.invert_yaxis()\n'
                                    '        unit = task_df["unit"].iloc[0]\n'
                                    '        ax.set_title(task_name)\n'
                                    '        ax.set_xlabel(f"Best model mean MAE '
                                    '({unit})")\n'
                                    '        for idx, (_, row) in '
                                    'enumerate(task_df.iterrows()):\n'
                                    '            ax.text(\n'
                                    '                row["mean_mae"],\n'
                                    '                idx,\n'
                                    '                f" {row[\'mean_mae\']:.3g} '
                                    '({row[\'model_display\']})",\n'
                                    '                va="center",\n'
                                    '                fontsize=8,\n'
                                    '            )\n'
                                    '\n'
                                    '    for ax in axes[len(tasks) :]:\n'
                                    '        ax.axis("off")\n'
                                    '\n'
                                    '    fig.suptitle("Composition-proxy vs '
                                    'structure-aware feature branches", y=1.02, '
                                    'fontsize=13)\n'
                                    '    return _save(fig, figures_dir / '
                                    '"05_structure_feature_branch_comparison.png")\n'
                                    '\n'
                                    '\n'
                                    'def save_all_figures(\n'
                                    '    metrics_df: pd.DataFrame,\n'
                                    '    summary_df: pd.DataFrame,\n'
                                    '    predictions_df: pd.DataFrame,\n'
                                    '    comparison_df: pd.DataFrame,\n'
                                    '    paths: RunPaths,\n'
                                    ') -> list[Path]:\n'
                                    '    figures = [\n'
                                    '        plot_model_mae_comparison(summary_df, '
                                    'paths.figures),\n'
                                    '        plot_fold_mae_distribution(metrics_df, '
                                    'paths.figures),\n'
                                    '        plot_tabpfn_vs_baseline(comparison_df, '
                                    'paths.figures),\n'
                                    '        plot_tabpfn_parity(predictions_df, '
                                    'paths.figures),\n'
                                    '        '
                                    'plot_structure_feature_set_comparison(summary_df, '
                                    'paths.figures),\n'
                                    '    ]\n'
                                    '    return [path for path in figures if path is '
                                    'not None]\n',
 'src/matbench_tabpfn/settings.py': '"""Shared project settings."""\n'
                                    '\n'
                                    'from __future__ import annotations\n'
                                    '\n'
                                    'from pathlib import Path\n'
                                    '\n'
                                    '\n'
                                    'PROJECT_ROOT = '
                                    'Path(__file__).resolve().parents[2]\n'
                                    '\n'
                                    'STARTER_TASKS = [\n'
                                    '    "matbench_steels",\n'
                                    '    "matbench_jdft2d",\n'
                                    '    "matbench_phonons",\n'
                                    '    "matbench_expt_gap",\n'
                                    ']\n'
                                    '\n'
                                    'DEFAULT_MODELS = [\n'
                                    '    "dummy_mean",\n'
                                    '    "ridge_cv",\n'
                                    '    "random_forest",\n'
                                    '    "extra_trees",\n'
                                    '    "hist_gradient_boosting",\n'
                                    '    "tabpfn",\n'
                                    ']\n'
                                    '\n'
                                    'MODEL_DISPLAY_NAMES = {\n'
                                    '    "dummy_mean": "Dummy mean",\n'
                                    '    "ridge_cv": "RidgeCV",\n'
                                    '    "random_forest": "Random forest",\n'
                                    '    "extra_trees": "Extra trees",\n'
                                    '    "hist_gradient_boosting": '
                                    '"HistGradientBoosting",\n'
                                    '    "tabpfn": "TabPFN",\n'
                                    '}\n'
                                    '\n'
                                    'MODEL_COLORS = {\n'
                                    '    "dummy_mean": "#8a8f98",\n'
                                    '    "ridge_cv": "#4c78a8",\n'
                                    '    "random_forest": "#59a14f",\n'
                                    '    "extra_trees": "#f28e2b",\n'
                                    '    "hist_gradient_boosting": "#e15759",\n'
                                    '    "tabpfn": "#6f4eae",\n'
                                    '}\n'
                                    '\n'
                                    'RANDOM_SEED = 42\n'
                                    'PRIMARY_METRIC = "mae"\n'}


def candidate_roots() -> list[Path]:
    candidates: list[Path] = []

    env_root = os.environ.get('MATBENCH_TABPFN_ROOT')
    if env_root:
        candidates.append(Path(env_root))

    vscode_notebook = globals().get('__vsc_ipynb_file__')
    if vscode_notebook:
        notebook_path = Path(vscode_notebook)
        candidates.extend([notebook_path.parent, notebook_path.parent.parent])

    cwd = Path.cwd()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / 'matbench-tabpfn',
        Path('/content/matbench-tabpfn'),
        Path('/content/drive/MyDrive/matbench-tabpfn'),
    ])

    seen = set()
    unique = []
    for candidate in candidates:
        try:
            resolved = candidate.expanduser().resolve()
        except OSError:
            continue
        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def is_project_root(path: Path) -> bool:
    return (
        (path / 'configs' / 'gpu_rerun.yml').exists()
        and (path / 'src' / 'matbench_tabpfn').is_dir()
    )


def sync_runtime_project(root: Path) -> Path:
    root = root.expanduser().resolve()
    for rel_path, contents in BOOTSTRAP_FILES.items():
        target = root / rel_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(contents, encoding='utf-8')
    os.environ['MATBENCH_TABPFN_ROOT'] = str(root)
    return root


def bootstrap_remote_project(root: Path = Path('/content/matbench-tabpfn')) -> Path:
    return sync_runtime_project(root)


def find_project_root() -> Path:
    checked = []
    for start in candidate_roots():
        for candidate in [start, *start.parents]:
            checked.append(candidate)
            if is_project_root(candidate):
                return candidate

    bootstrap_target = Path(os.environ.get('MATBENCH_TABPFN_ROOT', '/content/matbench-tabpfn'))
    bootstrap_root = bootstrap_remote_project(bootstrap_target)
    if is_project_root(bootstrap_root):
        print('Created remote runtime project at:', bootstrap_root)
        return bootstrap_root

    checked_text = '\n'.join(f'  - {path}' for path in dict.fromkeys(checked))
    raise RuntimeError(
        'Could not find or bootstrap the matbench-tabpfn project files on this GPU runtime.\n'
        'Set os.environ["MATBENCH_TABPFN_ROOT"] to a writable remote project folder and rerun this cell.\n\n'
        f'Checked paths:\n{checked_text}'
    )


PROJECT_ROOT = find_project_root()
# Keep remote runtime files in sync with this notebook. This fixes stale /content sources after notebook edits.
if str(PROJECT_ROOT).startswith('/content') or os.environ.get('MATBENCH_TABPFN_FORCE_SYNC') == '1':
    sync_runtime_project(PROJECT_ROOT)
    print('Synchronized runtime project files at:', PROJECT_ROOT)

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'gpu_rerun.yml'
if yaml is None:
    config = copy.deepcopy(DEFAULT_CONFIG)
    print('PyYAML is not installed; using the built-in default config for now.')
else:
    config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))

print('Project root:', PROJECT_ROOT)
print('Config:', CONFIG_PATH if yaml is not None else 'built-in defaults')
print('Tasks:', config['tasks'])
print('Models:', config['models'])


## 2. Verify CUDA GPU

This notebook is intentionally GPU-first. If CUDA is not available, fix the VSCode remote/kernel connection before starting the full run.

In [ ]:
import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is not available in this VSCode kernel.')

for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f'GPU {idx}: {torch.cuda.get_device_name(idx)} | {props.total_memory / 1024**3:.1f} GB')


## 3. Set TabPFN token without saving it

Do not hard-code the token into this notebook. Use an environment variable in the VSCode remote terminal, or paste it into the hidden prompt below.

In [ ]:
import getpass

if not os.environ.get('TABPFN_TOKEN'):
    os.environ['TABPFN_TOKEN'] = getpass.getpass('Paste TABPFN_TOKEN: ')

print('TABPFN_TOKEN is set:', bool(os.environ.get('TABPFN_TOKEN')))


## 4. Choose run settings

For a complete rerun, leave `DRY_RUN = False`. For a quick pipeline check, set `DRY_RUN = True` before running the expensive cells.

In [ ]:
import importlib
import matbench_tabpfn.paths as project_paths
importlib.reload(project_paths)
from matbench_tabpfn.paths import create_run_paths, collect_environment_manifest, write_json

DRY_RUN = False
RUN_ID = None  # None creates a timestamped run id. Or set, for example: 'gpu_full_rerun_01'.
CLEAN_EXISTING_RUN_DIR = bool(config['run'].get('clean_existing_run_dir', False))

TASKS = list(config['tasks'])
MODELS = list(config['models'])

if DRY_RUN:
    TASKS = ['matbench_steels']
    MODELS = ['dummy_mean', 'extra_trees']

# Your Google Drive project folder. In Colab this maps to /content/drive/MyDrive/...
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/MAT459/project')

# If this notebook is running on Colab, mount Drive so results persist next to the notebook.
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped:', exc)

# Priority:
# 1. Explicit environment variable MATBENCH_TABPFN_RESULTS_ROOT
# 2. Google Drive folder: /content/drive/MyDrive/MAT459 - project/results
# 3. Runtime project folder: PROJECT_ROOT/results
results_root_override = os.environ.get('MATBENCH_TABPFN_RESULTS_ROOT')
if results_root_override:
    RESULTS_ROOT = Path(results_root_override).expanduser().resolve()
elif Path('/content/drive/MyDrive').exists():
    RESULTS_ROOT = DRIVE_PROJECT_DIR / 'results'
    os.environ['MATBENCH_TABPFN_RESULTS_ROOT'] = str(RESULTS_ROOT)
else:
    configured_base_dir = Path(config['run']['base_dir'])
    RESULTS_ROOT = (PROJECT_ROOT / configured_base_dir.parent).resolve() if not configured_base_dir.is_absolute() else configured_base_dir.parent.resolve()

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_BASE_DIR = RESULTS_ROOT / 'runs'
LATEST_ROOT = RESULTS_ROOT

import inspect
create_run_paths_kwargs = dict(
    project_root=PROJECT_ROOT,
    run_id=RUN_ID,
    base_dir=RUNS_BASE_DIR,
    clean=CLEAN_EXISTING_RUN_DIR,
    update_latest=bool(config['run'].get('update_latest_pointer', True)),
)
if 'latest_root' in inspect.signature(create_run_paths).parameters:
    create_run_paths_kwargs['latest_root'] = LATEST_ROOT

paths = create_run_paths(**create_run_paths_kwargs)

manifest = collect_environment_manifest({
    'config_path': CONFIG_PATH,
    'tasks': TASKS,
    'models': MODELS,
    'dry_run': DRY_RUN,
    'results_root': RESULTS_ROOT,
    'config': config,
})
write_json(paths.logs / 'run_manifest.json', manifest)

print('Results root:', RESULTS_ROOT)
print('Run root:', paths.root)
print('Figures:', paths.figures)
print('Tasks:', TASKS)
print('Models:', MODELS)
paths.as_posix_dict()


## 5. Run official-fold experiments

This is the expensive cell. It saves partial metrics and predictions after every model, so if a later GPU run fails, earlier results are still available in the run directory.

In [ ]:
import importlib
import inspect

# If this is a Colab/remote runtime, make sure /content sources match this notebook's embedded source files.
if 'sync_runtime_project' in globals() and 'PROJECT_ROOT' in globals():
    sync_runtime_project(PROJECT_ROOT)

importlib.invalidate_caches()
import matbench_tabpfn.evaluation as project_evaluation
importlib.reload(project_evaluation)
from matbench_tabpfn.evaluation import run_official_fold_experiment

feature_sets = list(config['features'].get('feature_sets', [config['features'].get('feature_set', 'magpie')]))
if 'feature_sets' not in inspect.signature(run_official_fold_experiment).parameters:
    raise RuntimeError(
        'The remote runtime is still using an old matbench_tabpfn.evaluation module. '
        'Restart the kernel, rerun the first notebook cell, then rerun this cell.'
    )

metrics_df, predictions_df, summary_df, baseline_comparison_df, paired_comparison_df = run_official_fold_experiment(
    tasks=TASKS,
    models=MODELS,
    paths=paths,
    feature_sets=feature_sets,
    random_seed=int(config['run']['random_seed']),
    n_estimators=int(config['classical_models']['n_estimators']),
    tabpfn_n_estimators=int(config['tabpfn']['n_estimators']),
    tabpfn_device=str(config['tabpfn']['device']),
    tabpfn_predict_batch_size=int(config['tabpfn']['predict_batch_size']),
    use_feature_cache=bool(config['features']['use_cache']),
    show_feature_progress=bool(config['features']['show_progress']),
    feature_n_jobs=int(config['features'].get('n_jobs', 1)),
    continue_on_error=False,
    tabpfn_kwargs=dict(config['tabpfn'].get('kwargs', {})),
)

summary_df.sort_values(['task', 'rank_by_mae'])


## 6. Generate unified figures

All plots use the same style, model names, color mapping, DPI, and file naming. PNG and PDF versions are both saved.

In [ ]:
import importlib

# Force the remote runtime to use the plotting/analysis code embedded in this notebook.
if 'sync_runtime_project' in globals() and 'PROJECT_ROOT' in globals():
    sync_runtime_project(PROJECT_ROOT)

importlib.invalidate_caches()
import matbench_tabpfn.plotting as project_plotting
import matbench_tabpfn.analysis as project_analysis
importlib.reload(project_plotting)
importlib.reload(project_analysis)
from matbench_tabpfn.plotting import save_all_figures, PLOT_STYLE_VERSION
from matbench_tabpfn.analysis import create_report_artifacts

print('Plot style version:', PLOT_STYLE_VERSION)
print('Plotting module:', project_plotting.__file__)

# Delete old figures so the notebook cannot accidentally redisplay stale PNGs.
for pattern in ['*.png', '*.pdf']:
    for old_figure in paths.figures.glob(pattern):
        old_figure.unlink()

figure_paths = save_all_figures(
    metrics_df=metrics_df,
    summary_df=summary_df,
    predictions_df=predictions_df,
    comparison_df=baseline_comparison_df,
    paths=paths,
)

report_artifacts = create_report_artifacts(
    metrics_df=metrics_df,
    predictions_df=predictions_df,
    summary_df=summary_df,
    baseline_comparison_df=baseline_comparison_df,
    paired_comparison_df=paired_comparison_df,
    paths=paths,
    top_n_errors=15,
)

print('Figures:')
for path in figure_paths:
    print(' -', path)

print('\nReport artifacts:')
for name, path in report_artifacts.items():
    if name.endswith('_text'):
        continue
    print(f' - {name}: {path}')


## 7. Display results in this notebook

This cell reloads the saved CSV files and displays the main tables and figures inline. Run it after the figure generation cell, or rerun it later to inspect the latest saved run.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown


def find_latest_run_root() -> Path:
    if 'paths' in globals() and Path(paths.root).exists():
        return Path(paths.root)

    candidate_results = []
    if 'RESULTS_ROOT' in globals():
        candidate_results.append(Path(RESULTS_ROOT))
    if os.environ.get('MATBENCH_TABPFN_RESULTS_ROOT'):
        candidate_results.append(Path(os.environ['MATBENCH_TABPFN_RESULTS_ROOT']))
    candidate_results.extend([
        Path('/content/drive/MyDrive/MAT459 - project/results'),
        Path('/content/matbench-tabpfn/results'),
        PROJECT_ROOT / 'results' if 'PROJECT_ROOT' in globals() else Path.cwd() / 'results',
    ])

    for results_root in candidate_results:
        latest = results_root / 'latest'
        if latest.exists():
            return latest.resolve()
        latest_txt = results_root / 'latest_run.txt'
        if latest_txt.exists():
            return Path(latest_txt.read_text().strip())
        runs_dir = results_root / 'runs'
        if runs_dir.exists():
            runs = [p for p in runs_dir.iterdir() if p.is_dir()]
            if runs:
                return sorted(runs, key=lambda p: p.stat().st_mtime)[-1]

    raise FileNotFoundError('No saved run found. Run the experiment and figure-generation cells first.')


def read_csv_if_exists(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() and path.stat().st_size > 0 else pd.DataFrame()


RUN_ROOT = find_latest_run_root()
METRICS_DIR = RUN_ROOT / 'metrics'
TABLES_DIR = RUN_ROOT / 'tables'
FIGURES_DIR = RUN_ROOT / 'figures'

summary = read_csv_if_exists(METRICS_DIR / 'model_summary.csv')
baseline = read_csv_if_exists(METRICS_DIR / 'best_baseline_comparison.csv')
paired = read_csv_if_exists(METRICS_DIR / 'paired_fold_comparisons.csv')
structure_branch = read_csv_if_exists(TABLES_DIR / 'structure_feature_branch_summary.csv')
top_errors = read_csv_if_exists(TABLES_DIR / 'top_absolute_errors.csv')
tabpfn_errors = read_csv_if_exists(TABLES_DIR / 'tabpfn_vs_best_baseline_sample_errors.csv')
auto_summary_path = TABLES_DIR / 'auto_summary.md'

print('Run root:', RUN_ROOT)
print('Figures dir:', FIGURES_DIR)

if auto_summary_path.exists():
    display(Markdown(auto_summary_path.read_text()))

if not summary.empty:
    display(Markdown('### Model summary'))
    display(summary.sort_values(['task', 'rank_by_mae']))

if not structure_branch.empty:
    display(Markdown('### Structure-aware branch vs composition proxy'))
    display(structure_branch.sort_values('task'))

if not baseline.empty:
    display(Markdown('### TabPFN vs best non-TabPFN baseline'))
    display(baseline.query("model == 'tabpfn'").sort_values(['task', 'feature_set']))

if not paired.empty:
    display(Markdown('### Paired fold comparison'))
    display(paired.sort_values(['task', 'reference_feature_set']))

if not top_errors.empty:
    display(Markdown("### Top absolute errors for each task best model"))
    display(top_errors.sort_values(['task', 'absolute_error'], ascending=[True, False]).head(60))

if not tabpfn_errors.empty:
    display(Markdown('### Samples where TabPFN loses most to the best baseline'))
    display(tabpfn_errors.sort_values('tabpfn_error_minus_baseline_error', ascending=False).head(60))

png_files = sorted(FIGURES_DIR.glob('*.png'))
if not png_files:
    print('No PNG figures found. Run the figure-generation cell first.')
else:
    display(Markdown('### Figures'))
    for fig_path in png_files:
        display(Markdown(f'#### {fig_path.name}'))
        display(Image(filename=str(fig_path)))


## 8. Package outputs

The zip file contains metrics, predictions, features, logs, tables, and figures for this exact run.

In [ ]:
import shutil

zip_path = shutil.make_archive(str(paths.root), 'zip', paths.root)
print('Created:', zip_path)


## Reporting notes

Use `model_summary.csv` for the main results table. Use `best_baseline_comparison.csv` to state whether TabPFN beats the strongest non-TabPFN baseline. For `matbench_jdft2d` and `matbench_phonons`, report the feature source as `composition_from_structure`; these are composition-proxy results, not full structure-aware models.